In [47]:
# ============================================================
# 043_weekly_targets_review
# ============================================================
#
# Overview
# ----------------
# Weekly checkpoint to review and tune all monitoring targets (VC, Startup, Policy, People).
# This notebook:
#   - summarizes target performance over the last N days using linked Events
#   - estimates signal vs noise per target (metrics + optional LLM judgment)
#   - proposes concrete updates to Target configuration (Priority / Cadence / Status / Keywords / URLs)
#   - writes reviewable proposal pages to the Weekly Target Update DB (human-in-the-loop)
#   - exports lightweight charts + CSV + a text report for the weekly review packet
#
# The primary goal is to keep the monitoring set sharp:
# reduce noisy targets, upgrade consistently high-signal targets, and tune search inputs
# (keywords / source URLs / source type) when retrieval quality drifts.
#
# Inputs / Outputs
# ----------------
# Inputs (env.txt):
#   - NOTION_TOKEN
#   - NOTION_VERSION (default: 2025-09-03)
#   - NOTION_MONITORING_TARGETS_DB_ID
#   - NOTION_EVENTS_DB_ID
#   - NOTION_WEEKLY_TARGET_UPDATE_DB_ID
#   - (optional) NOTION_WEEKLY_DIGESTS_DB_ID  # used to resolve current Week page automatically
#   - (optional) GOOGLE_CSE_API_KEY / GOOGLE_CSE_CX
#   - (optional) NEWSAPI_KEY
#   - OPENAI_API_KEY
#
# Outputs:
#   - Weekly Target Update DB:
#       Proposal pages (per target × per field) with:
#         Field, Current Value, Proposed Value, Change Summary, Rationale, Confidence, Evidence links
#   - Local artifacts (outputs/weekly_target_review/):
#       - PNG charts (effectiveness, noise, proposals by type, etc.)
#       - CSV exports (evaluation summary, proposals)
#       - Text report (weekly_target_review_report_{current_week}.txt)
#
# Structure
# ----------------
# Cell 01: Imports and environment setup (env.txt, OpenAI client)
# Cell 02: Notion HTTP client init (headers + GET/POST/PATCH helpers)
# Cell 03: Resolve and cache data_source_id for required Notion DBs (ONE-TIME scan)
# Cell 04: Introspect DB schemas (type-only inference if properties is empty)
# Cell 05: Load active targets from Monitoring Targets DB (Enabled=true, Status!=Archived)
# Cell 06: Load Events for lookback window and extract Target relations
# Cell 07: Aggregate Events → per-target metrics (volume / diversity / recency / dedup)
# Cell 08: (Optional) LLM evaluation of target effectiveness + refined noise ratio
# Cell 09: Identify priority adjustments and removal candidates (rules + thresholds)
# Cell 10: Identify cadence/status changes (rules + thresholds)
# Cell 11: Target tuning suggestions (Keywords / Source URLs) using NewsAPI + Google CSE + LLM
# Cell 12: Write proposal pages to Weekly Target Update DB (no auto-apply)
# Cell 13: (Optional) Apply approved changes back to Monitoring Targets DB (off by default)
# Cell 13.5: (Optional) Rebuild proposals_df from Weekly Target Update DB (this week) for reporting
# Cell 14: Summary export (single-chart PNGs + CSV + text report; no subplots)
#
# Notes
# ----------------
# - Notion API requirements:
#   * Assume Notion API version >= 2025-09-03.
#   * IDs in env.txt are database IDs (UUID).
#   * DO NOT use POST /v1/databases/{database_id}/query (unsupported in this environment).
#   * For metadata/schema: GET /v1/databases/{database_id}
#   * For content queries: POST /v1/data_sources/{data_source_id}/query only.
#   * Because database responses may not expose data_source_id in a single fixed field:
#       1) GET /v1/databases/{database_id}
#       2) recursively scan JSON for UUID-like strings
#       3) validate candidates via POST /v1/data_sources/{candidate}/query {"page_size": 1}
#       4) cache the first working candidate as data_source_id
#   * Resolve data_source_id ONCE in Cell 03 and reuse it everywhere (no repeated deep-scan).
#
# - Schema inference:
#   * If database.properties is empty, infer types by sampling one page (page_size=1).
#   * In inference mode, only the "type" field is trusted (do not infer select options/relation targets).
#
# - Human-in-the-loop:
#   * This notebook writes proposals for review; it does NOT automatically change targets by default.
#   * Auto-apply (Cell 13) should be gated behind an explicit flag and/or proposal Status=Approved.
#
# - Target-type-aware tuning:
#   * Do NOT assume .gov is always preferred.
#   * POLICY: government/regulatory sources often preferred when relevant.
#   * VC/STARTUP/PEOPLE: prioritize official sites + high-signal primary sources for that domain.
#
# - UUID normalization:
#   * Always normalize 32-hex IDs into hyphenated UUIDs before calling Notion endpoints.
#
# - Default filtering:
#   * Targets: Enabled=true AND Status != "Archived"
#   * Events: within the last DAYS_LOOKBACK days


In [3]:
# ============================================================
# Cell 01 — Imports and environment setup
# ============================================================
# Overview:
#   Load env variables from env.txt, import core libraries, and define tiny utilities
#   used across the notebook (UUID normalization, time helpers).
#
# Inputs / Outputs:
#   Inputs:  env.txt (dotenv)
#   Outputs: global constants (NOTION_TOKEN, NOTION_VERSION, DB IDs), helper functions
#
# Notes:
#   - Do NOT create or modify env.txt in this notebook.
#   - Notion querying is constrained to /v1/data_sources/{data_source_id}/query (handled later).
#

from dotenv import load_dotenv
import os
import re
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Any, Optional, Tuple

import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

# --- Mandatory env loading ---
load_dotenv("env.txt")

# --- Environment variables (SPEC-ALIGNED) ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
NOTION_TOKEN = os.getenv("NOTION_TOKEN")
NOTION_VERSION = os.getenv("NOTION_VERSION", "2025-09-03")

EVENTS_DB_ID = os.getenv("NOTION_EVENTS_DB_ID")
MONITORING_TARGETS_DB_ID = os.getenv("NOTION_MONITORING_TARGETS_DB_ID")

# --- Timezone (JST) ---
JST = timezone(timedelta(hours=9))

# --- Minimal runtime config (if needed later) ---
LLM_PROVIDER = "OpenAI"
LLM_MODEL = "gpt-4o-mini"
LLM_TEMPERATURE = 0.0

# --- Helper: normalize UUID ---
def normalize_uuid(uuid_str: Optional[str]) -> Optional[str]:
    """
    Normalize UUID-like strings into hyphenated UUID form.
    Accepts:
      - 32 hex chars (with or without separators)
      - already-hyphenated UUID
    Returns the original if not convertible.
    """
    if not uuid_str:
        return uuid_str
    s = str(uuid_str).strip().lower()
    clean = re.sub(r"[^a-f0-9]", "", s)
    if len(clean) == 32:
        return f"{clean[:8]}-{clean[8:12]}-{clean[12:16]}-{clean[16:20]}-{clean[20:]}"
    return uuid_str

# Normalize DB IDs early (harmless even if already hyphenated)
EVENTS_DB_ID = normalize_uuid(EVENTS_DB_ID)
MONITORING_TARGETS_DB_ID = normalize_uuid(MONITORING_TARGETS_DB_ID)

# --- Validate required variables for THIS notebook ---
required_vars = {
    "NOTION_TOKEN": NOTION_TOKEN,
    "NOTION_EVENTS_DB_ID": EVENTS_DB_ID,
    "NOTION_MONITORING_TARGETS_DB_ID": MONITORING_TARGETS_DB_ID,
}
missing = [k for k, v in required_vars.items() if not v]

if missing:
    raise RuntimeError(
        "Missing required environment variables in env.txt: "
        + ", ".join(missing)
        + "\nPlease set them and re-run Cell 01."
    )

print("✓ Environment and imports loaded successfully")
print(f"  NOTION_VERSION: {NOTION_VERSION}")
print(f"  EVENTS_DB_ID: {EVENTS_DB_ID}")
print(f"  MONITORING_TARGETS_DB_ID: {MONITORING_TARGETS_DB_ID}")
print(f"  LLM (optional): {LLM_PROVIDER} / {LLM_MODEL} @ temp={LLM_TEMPERATURE}")


✓ Environment and imports loaded successfully
  NOTION_VERSION: 2025-09-03
  EVENTS_DB_ID: 2f08e0e4-d162-80be-b40c-f607bbb3b828
  MONITORING_TARGETS_DB_ID: 2f08e0e4-d162-8013-b322-cb01d78f0de8
  LLM (optional): OpenAI / gpt-4o-mini @ temp=0.0


In [4]:
# ============================================================
# Cell 02 — Notion HTTP client and basic connectivity check
# ============================================================
# Overview:
#   Initialize low-level Notion HTTP helpers and verify that the integration
#   can access the target databases via metadata requests.
#
# Inputs / Outputs:
#   Inputs:  NOTION_TOKEN, NOTION_VERSION, DB IDs (from Cell 01)
#   Outputs: notion_headers, notion_get(), notion_post()
#
# Notes:
#   - This cell does NOT resolve data_source_id.
#   - No database querying is performed here.
#   - /v1/databases/{database_id} is used for connectivity checks.
#

# --- Safety checks ---
if not NOTION_TOKEN:
    raise RuntimeError(
        "NOTION_TOKEN is not set. Please define it in env.txt and re-run Cell 01."
    )

if not NOTION_VERSION:
    raise RuntimeError("NOTION_VERSION must be defined.")

# --- Base Notion API headers ---
notion_headers = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}

NOTION_API_BASE = "https://api.notion.com"

# --- HTTP helpers ---
def notion_get(endpoint: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """
    Perform GET request against Notion API.
    """
    url = f"{NOTION_API_BASE}{endpoint}"
    resp = requests.get(url, headers=notion_headers, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

def notion_post(endpoint: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    """
    Perform POST request against Notion API.
    """
    url = f"{NOTION_API_BASE}{endpoint}"
    resp = requests.post(url, headers=notion_headers, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

# --- Connectivity check via database metadata ---
def _check_database_access(db_id: str, label: str) -> None:
    if not db_id:
        raise RuntimeError(f"{label} DB ID is missing.")
    try:
        meta = notion_get(f"/v1/databases/{db_id}")
        title = meta.get("title", [])
        title_text = (
            title[0]["plain_text"] if title and "plain_text" in title[0] else "Untitled"
        )
        print(f"✓ Access OK: {label} DB — {title_text}")
    except requests.HTTPError as e:
        raise RuntimeError(
            f"Failed to access {label} DB metadata ({db_id}). "
            f"Check integration permissions and DB ID.\nOriginal error: {e}"
        )

# --- Run checks ---
_check_database_access(EVENTS_DB_ID, "Events")
_check_database_access(MONITORING_TARGETS_DB_ID, "Monitoring Targets")

print("✓ Notion HTTP client initialized")
print(f"  Notion-Version: {NOTION_VERSION}")


✓ Access OK: Events DB — EVENTS_DB
✓ Access OK: Monitoring Targets DB — MONITORING_TARGETS_DB
✓ Notion HTTP client initialized
  Notion-Version: 2025-09-03


In [33]:
# ============================================================
# Cell 03 — Resolve data_source_id (ONE-TIME deep scan + validation + cache)
# ============================================================
# Overview:
#   Resolve data_source_id for each Notion database ID (UUID) using the required flow:
#     1) GET /v1/databases/{database_id}
#     2) recursively scan the JSON for UUID-like strings
#     3) validate candidates via POST /v1/data_sources/{candidate}/query {"page_size": 1}
#     4) pick the first candidate that succeeds
#   Cache results in RESOLVED_DB and never deep-scan again in later cells.
#
# Inputs / Outputs:
#   Inputs:  notion_get, notion_post, normalize_uuid, EVENTS_DB_ID, MONITORING_TARGETS_DB_ID
#   Outputs: RESOLVED_DB = { name: {"database_id": ..., "data_source_id": ...}, ... }
#
# Notes:
#   - Do NOT use POST /v1/databases/{database_id}/query (unsupported here).
#   - Weekly Target Update DB is optional for 043; resolve only if present in env.
#

# --- Optional DB (do not fail if missing) ---
WEEKLY_TARGET_UPDATE_DB_ID = normalize_uuid(os.getenv("NOTION_WEEKLY_TARGET_UPDATE_DB_ID"))

# --- Global cache (created once) ---
RESOLVED_DB: Dict[str, Dict[str, str]] = {}

UUID_RE = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$", re.I)
HEX32_RE = re.compile(r"^[0-9a-f]{32}$", re.I)

def _looks_like_uuid(s: str) -> bool:
    if not s:
        return False
    s = str(s).strip().lower()
    if UUID_RE.match(s):
        return True
    if HEX32_RE.match(re.sub(r"[^0-9a-f]", "", s)):
        return True
    return False

def _scan_for_uuid_candidates(obj: Any) -> List[str]:
    """
    Recursively scan any JSON-like structure for UUID-like strings.
    Returns a de-duplicated list (stable order).
    """
    found: List[str] = []
    seen = set()

    def _visit(x: Any):
        if isinstance(x, dict):
            for k, v in x.items():
                _visit(k)
                _visit(v)
        elif isinstance(x, list):
            for item in x:
                _visit(item)
        elif isinstance(x, str):
            s = x.strip()
            if _looks_like_uuid(s):
                u = normalize_uuid(s)
                if u and u not in seen:
                    seen.add(u)
                    found.append(u)

    _visit(obj)
    return found

def _validate_data_source_id(candidate_uuid: str) -> bool:
    """
    Validate candidate by attempting a minimal query:
      POST /v1/data_sources/{candidate}/query {"page_size": 1}
    Success => candidate is a valid data_source_id.
    """
    try:
        _ = notion_post(f"/v1/data_sources/{candidate_uuid}/query", {"page_size": 1})
        return True
    except requests.HTTPError:
        return False
    except requests.RequestException:
        return False

def resolve_data_source_id_once(db_id: str, name: str) -> Dict[str, str]:
    """
    Resolve a data_source_id for a given database_id using the mandated approach.
    Returns {"database_id": ..., "data_source_id": ...}
    """
    if name in RESOLVED_DB:
        return RESOLVED_DB[name]

    db_id = normalize_uuid(db_id)
    if not db_id:
        raise RuntimeError(f"{name}: database_id is missing.")

    # 1) fetch database metadata
    meta = notion_get(f"/v1/databases/{db_id}")

    # 2) scan for UUID-like strings
    candidates = _scan_for_uuid_candidates(meta)

    # Heuristic: try the most plausible first by bringing db_id to front if present
    # (db_id itself will fail validation; this is OK—just a small ordering tweak)
    if db_id in candidates:
        candidates = [db_id] + [c for c in candidates if c != db_id]

    # 3) validate via data_sources query
    resolved: Optional[str] = None
    for c in candidates:
        if _validate_data_source_id(c):
            resolved = c
            break

    if not resolved:
        raise RuntimeError(
            f"{name}: failed to resolve data_source_id.\n"
            f"- database_id: {db_id}\n"
            f"- scanned candidates: {len(candidates)}\n"
            "Ensure the integration has access and that this environment supports data_sources querying."
        )

    RESOLVED_DB[name] = {"database_id": db_id, "data_source_id": resolved}
    return RESOLVED_DB[name]

print("Resolving data_source_id (deep scan happens ONLY in this cell)...")

# --- Required DBs ---
RESOLVED_DB["events"] = resolve_data_source_id_once(EVENTS_DB_ID, "events")
RESOLVED_DB["monitoring_targets"] = resolve_data_source_id_once(MONITORING_TARGETS_DB_ID, "monitoring_targets")

# --- Optional DB (only if provided) ---
if WEEKLY_TARGET_UPDATE_DB_ID:
    try:
        RESOLVED_DB["weekly_target_update"] = resolve_data_source_id_once(
            WEEKLY_TARGET_UPDATE_DB_ID, "weekly_target_update"
        )
    except Exception as e:
        # Do not fail the notebook for 043, but surface the issue clearly
        print("⚠️  weekly_target_update DB provided but could not resolve data_source_id.")
        print(f"    Reason: {e}")

print("\n✓ Resolved data_source_id cache ready:")
for k, v in RESOLVED_DB.items():
    print(f"  - {k}: database_id={v['database_id']} | data_source_id={v['data_source_id']}")


# ============================================================
# Cell 03A — Resolve data_source_id for Weekly Digests DB (additional)
# ============================================================
# Overview:
#   Add Weekly Digests DB to RESOLVED_DB by resolving its data_source_id
#   using the same "deep scan -> validate via data_sources query" approach.
#
# Inputs / Outputs:
#   Inputs:  NOTION_WEEKLY_DIGESTS_DB_ID (env), notion_get, notion_post, normalize_uuid
#   Outputs: RESOLVED_DB["weekly_digests"] = {"database_id": ..., "data_source_id": ...}
#

import re
import json

WEEKLY_DIGESTS_DB_ID = normalize_uuid(os.getenv("NOTION_WEEKLY_DIGESTS_DB_ID", ""))
if not WEEKLY_DIGESTS_DB_ID:
    raise RuntimeError("NOTION_WEEKLY_DIGESTS_DB_ID is not set in env.txt")

# Ensure RESOLVED_DB exists
if "RESOLVED_DB" not in globals() or RESOLVED_DB is None:
    RESOLVED_DB = {}

def _uuid_like_candidates(obj: Any) -> list[str]:
    """
    Recursively scan JSON for UUID-like strings, normalize to hyphenated UUID.
    We accept:
      - 32 hex
      - hyphenated UUID
    """
    found = []
    if isinstance(obj, dict):
        for v in obj.values():
            found.extend(_uuid_like_candidates(v))
    elif isinstance(obj, list):
        for v in obj:
            found.extend(_uuid_like_candidates(v))
    elif isinstance(obj, str):
        s = obj.strip()
        # collect both hyphenated and 32-hex
        m = re.findall(r"[a-fA-F0-9]{32}|[a-fA-F0-9]{8}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{12}", s)
        for x in m:
            nx = normalize_uuid(x)
            if nx:
                found.append(nx)
    return found

def _validate_data_source_id(candidate_id: str) -> bool:
    """
    Validate by trying POST /v1/data_sources/{id}/query with page_size=1.
    Success => True.
    """
    try:
        _ = notion_post(f"/v1/data_sources/{candidate_id}/query", {"page_size": 1})
        return True
    except requests.exceptions.HTTPError as e:
        return False
    except Exception:
        return False

def resolve_data_source_id_via_scan(database_id: str, name: str) -> str:
    """
    Required by your environment constraints:
      1) GET /v1/databases/{database_id}
      2) deep scan JSON for UUID-like strings
      3) validate candidates via POST /v1/data_sources/{candidate}/query
      4) pick first success
    """
    db_meta = notion_get(f"/v1/databases/{database_id}")
    candidates = _uuid_like_candidates(db_meta)

    # de-dup while preserving order
    seen = set()
    uniq = []
    for c in candidates:
        if c in seen:
            continue
        seen.add(c)
        uniq.append(c)

    print(f"[{name}] scanned UUID-like candidates: {len(uniq)}")

    for c in uniq:
        if _validate_data_source_id(c):
            print(f"[{name}] ✓ data_source_id resolved: {c}")
            return c

    raise RuntimeError(f"[{name}] Failed to resolve data_source_id from database metadata scan.")

# Resolve & register
weekly_digests_ds = resolve_data_source_id_via_scan(WEEKLY_DIGESTS_DB_ID, "weekly_digests")

RESOLVED_DB["weekly_digests"] = {
    "database_id": WEEKLY_DIGESTS_DB_ID,
    "data_source_id": weekly_digests_ds,
}

print("✓ RESOLVED_DB updated with weekly_digests")



Resolving data_source_id (deep scan happens ONLY in this cell)...

✓ Resolved data_source_id cache ready:
  - events: database_id=2f08e0e4-d162-80be-b40c-f607bbb3b828 | data_source_id=2f08e0e4-d162-8073-8735-000bf6f8f8dd
  - monitoring_targets: database_id=2f08e0e4-d162-8013-b322-cb01d78f0de8 | data_source_id=2f08e0e4-d162-80a5-a93a-000bc58517a2
  - weekly_target_update: database_id=2ff8e0e4-d162-80c0-b934-c9639ca5069a | data_source_id=2ff8e0e4-d162-8041-a4a4-000bcc5e745e
[weekly_digests] scanned UUID-like candidates: 4
[weekly_digests] ✓ data_source_id resolved: 2ff8e0e4-d162-8075-a3c1-000b4315606c
✓ RESOLVED_DB updated with weekly_digests


In [17]:
# ============================================================
# Cell 04 — Schema introspection and validation (metadata-first, inference via sample page)
# ============================================================
# Overview:
#   Introspect Notion database properties via GET /v1/databases/{database_id}.
#   If properties are empty, infer property "type" ONLY by sampling one page via
#   POST /v1/data_sources/{data_source_id}/query (page_size=1).
#   Optionally validate against declared schemas (EVENTS_SCHEMA, MONITORING_TARGETS_SCHEMA)
#   if they exist in the runtime.
#
# Inputs / Outputs:
#   Inputs:  RESOLVED_DB, notion_get, notion_post
#   Outputs: DB_META, DB_PROPERTIES, DB_PROP_TYPES, SCHEMA_REPORT
#
# Notes:
#   - Do NOT use /v1/search for schema inference.
#   - In inference mode, only "type" is trusted.
#   - Validation is skipped if schema dicts are not defined yet.
#

DB_META: Dict[str, Dict[str, Any]] = {}
DB_PROPERTIES: Dict[str, Dict[str, Any]] = {}
DB_PROP_TYPES: Dict[str, Dict[str, str]] = {}
SCHEMA_REPORT: Dict[str, Any] = {}

def _data_source_query_one(data_source_id: str) -> Dict[str, Any]:
    return notion_post(f"/v1/data_sources/{data_source_id}/query", {"page_size": 1})

def _infer_property_types_from_sample_page(sample_page: Dict[str, Any]) -> Dict[str, str]:
    props = (sample_page or {}).get("properties", {}) or {}
    inferred: Dict[str, str] = {}
    for prop_name, prop_obj in props.items():
        if isinstance(prop_obj, dict):
            t = prop_obj.get("type")
            if isinstance(t, str) and t:
                inferred[prop_name] = t
    return inferred

def introspect_db(name: str) -> None:
    if name not in RESOLVED_DB:
        raise RuntimeError(f"RESOLVED_DB missing key: {name}")

    database_id = RESOLVED_DB[name]["database_id"]
    data_source_id = RESOLVED_DB[name]["data_source_id"]

    meta = notion_get(f"/v1/databases/{database_id}")
    DB_META[name] = meta

    properties = meta.get("properties", {}) or {}
    DB_PROPERTIES[name] = properties

    if properties:
        types = {}
        for pn, po in properties.items():
            if isinstance(po, dict) and isinstance(po.get("type"), str):
                types[pn] = po["type"]
        DB_PROP_TYPES[name] = types
        print(f"✓ {name}: metadata properties(types)={len(types)}")
        return

    print(f"⚠ {name}: database.properties is empty; inferring types from one sample page...")
    q = _data_source_query_one(data_source_id)
    results = q.get("results", []) or []
    if not results:
        DB_PROP_TYPES[name] = {}
        print(f"  ✗ {name}: could not sample any page (empty data source).")
        return

    inferred = _infer_property_types_from_sample_page(results[0])
    DB_PROP_TYPES[name] = inferred
    print(f"  ✓ {name}: inferred types from sample page = {len(inferred)} (type-only trust)")

def validate_against_declared_schema(name: str, declared_schema: Dict[str, Any]) -> Dict[str, Any]:
    required = (declared_schema or {}).get("required_properties", {}) or {}
    actual_types = DB_PROP_TYPES.get(name, {}) or {}

    missing = []
    mismatches = []
    for prop, expected_type in required.items():
        if prop not in actual_types:
            missing.append(prop)
        else:
            actual = actual_types[prop]
            if expected_type != actual:
                mismatches.append({"property": prop, "expected": expected_type, "actual": actual})

    return {
        "missing_required_properties": missing,
        "type_mismatches": mismatches,
        "required_count": len(required),
        "actual_count": len(actual_types),
        "inference_mode": (len(DB_PROPERTIES.get(name, {}) or {}) == 0),
    }

def _print_type_snapshot(name: str, types_map: Dict[str, str], limit: int = 50) -> None:
    items = sorted(types_map.items(), key=lambda x: x[0])
    print(f"  properties(types) snapshot ({min(len(items), limit)}/{len(items)}):")
    for k, v in items[:limit]:
        print(f"    - {k}: {v}")

print("Introspecting database schemas...\n")

introspect_db("monitoring_targets")
introspect_db("events")
if "weekly_target_update" in RESOLVED_DB:
    introspect_db("weekly_target_update")

print("\nValidating against declared schemas...\n")

# --- SAFE lookup: schemas may not exist yet depending on cell order ---
monitoring_schema = globals().get("MONITORING_TARGETS_SCHEMA")
events_schema = globals().get("EVENTS_SCHEMA")

if monitoring_schema is not None:
    SCHEMA_REPORT["monitoring_targets"] = validate_against_declared_schema("monitoring_targets", monitoring_schema)
else:
    SCHEMA_REPORT["monitoring_targets"] = {"note": "MONITORING_TARGETS_SCHEMA is not defined; validation skipped."}

if events_schema is not None:
    SCHEMA_REPORT["events"] = validate_against_declared_schema("events", events_schema)
else:
    SCHEMA_REPORT["events"] = {"note": "EVENTS_SCHEMA is not defined; validation skipped."}

if "weekly_target_update" in RESOLVED_DB:
    SCHEMA_REPORT["weekly_target_update"] = {
        "note": "No declared schema enforced here (optional DB for 043).",
        "actual_count": len(DB_PROP_TYPES.get("weekly_target_update", {}) or {}),
        "inference_mode": (len(DB_PROPERTIES.get("weekly_target_update", {}) or {}) == 0),
    }

def _print_report(db_key: str, report: Dict[str, Any]) -> None:
    print(f"--- {db_key} ---")
    if "note" in report:
        print(f"  {report['note']}")
        _print_type_snapshot(db_key, DB_PROP_TYPES.get(db_key, {}) or {})
        return

    print(f"  required: {report['required_count']} | actual(types): {report['actual_count']} | inference_mode={report['inference_mode']}")
    if report["missing_required_properties"]:
        print(f"  ⚠ Missing required: {report['missing_required_properties']}")
    if report["type_mismatches"]:
        print("  ⚠ Type mismatches:")
        for m in report["type_mismatches"]:
            print(f"    - {m['property']}: expected={m['expected']} actual={m['actual']}")
    if (not report["missing_required_properties"]) and (not report["type_mismatches"]):
        print("  ✓ Schema looks compatible")
    _print_type_snapshot(db_key, DB_PROP_TYPES.get(db_key, {}) or {})

for k, rep in SCHEMA_REPORT.items():
    _print_report(k, rep)

print("\n✓ Schema introspection + (optional) validation complete")


Introspecting database schemas...

⚠ monitoring_targets: database.properties is empty; inferring types from one sample page...
  ✓ monitoring_targets: inferred types from sample page = 14 (type-only trust)
⚠ events: database.properties is empty; inferring types from one sample page...
  ✓ events: inferred types from sample page = 15 (type-only trust)
⚠ weekly_target_update: database.properties is empty; inferring types from one sample page...
  ✓ weekly_target_update: inferred types from sample page = 14 (type-only trust)

Validating against declared schemas...

--- monitoring_targets ---
  MONITORING_TARGETS_SCHEMA is not defined; validation skipped.
  properties(types) snapshot (14/14):
    - Cadence: select
    - Enabled: checkbox
    - Error Count: number
    - Last Checked: date
    - Last Error: rich_text
    - Name: title
    - Next Check: date
    - Notes: rich_text
    - Priority: select
    - Search Keywords: rich_text
    - Source Type: select
    - Source URLs: rich_text
  

In [18]:
# ============================================================
# Cell 05 — Fetch all active monitoring targets (data_sources query only)
# ============================================================
# Overview:
#   Fetch monitoring targets from Notion via POST /v1/data_sources/{data_source_id}/query
#   (pagination supported). Prefer server-side filters when possible; always apply
#   a defensive local filter (Enabled=True and Status != 'Archived').
#
# Inputs / Outputs:
#   Inputs:  RESOLVED_DB["monitoring_targets"]["data_source_id"], DB_PROP_TYPES
#   Outputs: targets_df (pandas DataFrame), raw_targets (list of Notion pages)
#
# Notes:
#   - DO NOT use /v1/search.
#   - Property extraction is type-driven using DB_PROP_TYPES (inferred or metadata).
#

MONITORING_DS_ID = RESOLVED_DB["monitoring_targets"]["data_source_id"]

# --- Property extraction helpers (type-safe, multi-fragment aware) ---

def _join_rich_text(arr: List[Dict[str, Any]]) -> str:
    if not arr:
        return ""
    parts = []
    for x in arr:
        t = x.get("plain_text")
        if isinstance(t, str) and t:
            parts.append(t)
    return "".join(parts).strip()

def extract_property(prop_obj: Optional[Dict[str, Any]], prop_type: Optional[str]) -> Any:
    """
    Extract a normalized Python value from a Notion property object, using its type.
    Returns:
      - title/rich_text: str
      - select: str
      - multi_select: List[str]
      - checkbox: bool
      - number: float|int|None
      - date: str|None (start ISO)
      - relation: List[str] (page IDs, normalized)
      - url/email/phone_number: str
      - people: List[str] (ids)
      - default: None
    """
    if not prop_obj or not prop_type:
        return None

    t = prop_type

    if t == "title":
        return _join_rich_text(prop_obj.get("title", []) or [])
    if t == "rich_text":
        return _join_rich_text(prop_obj.get("rich_text", []) or [])
    if t == "select":
        s = prop_obj.get("select")
        return (s or {}).get("name") if isinstance(s, dict) else None
    if t == "multi_select":
        arr = prop_obj.get("multi_select", []) or []
        return [x.get("name") for x in arr if isinstance(x, dict) and x.get("name")]
    if t == "checkbox":
        return bool(prop_obj.get("checkbox", False))
    if t == "number":
        return prop_obj.get("number")
    if t == "date":
        d = prop_obj.get("date")
        if isinstance(d, dict):
            return d.get("start")
        return None
    if t == "relation":
        rels = prop_obj.get("relation", []) or []
        ids = []
        for r in rels:
            if isinstance(r, dict) and r.get("id"):
                ids.append(normalize_uuid(r["id"]))
        return ids
    if t == "url":
        return prop_obj.get("url")
    if t == "email":
        return prop_obj.get("email")
    if t == "phone_number":
        return prop_obj.get("phone_number")
    if t == "people":
        arr = prop_obj.get("people", []) or []
        return [x.get("id") for x in arr if isinstance(x, dict) and x.get("id")]

    return None

# --- Data source query helpers ---

def notion_query_data_source_page(
    data_source_id: str,
    *,
    filter_obj: Optional[Dict[str, Any]] = None,
    sorts: Optional[List[Dict[str, Any]]] = None,
    page_size: int = 100,
    start_cursor: Optional[str] = None
) -> Dict[str, Any]:
    payload: Dict[str, Any] = {"page_size": page_size}
    if filter_obj:
        payload["filter"] = filter_obj
    if sorts:
        payload["sorts"] = sorts
    if start_cursor:
        payload["start_cursor"] = start_cursor
    return notion_post(f"/v1/data_sources/{data_source_id}/query", payload)

def notion_query_data_source_all(
    data_source_id: str,
    *,
    filter_obj: Optional[Dict[str, Any]] = None,
    sorts: Optional[List[Dict[str, Any]]] = None,
    page_size: int = 100,
    max_pages: int = 50
) -> List[Dict[str, Any]]:
    results: List[Dict[str, Any]] = []
    cursor: Optional[str] = None
    for _ in range(max_pages):
        resp = notion_query_data_source_page(
            data_source_id,
            filter_obj=filter_obj,
            sorts=sorts,
            page_size=page_size,
            start_cursor=cursor
        )
        batch = resp.get("results", []) or []
        results.extend(batch)
        if not resp.get("has_more"):
            break
        cursor = resp.get("next_cursor")
        if not cursor:
            break
    return results

# --- Build preferred server-side filter (best-effort) ---
# If your Notion environment rejects this filter structure, the query will error;
# we fall back to unfiltered query + local filtering.
preferred_filter = {
    "and": [
        {"property": "Enabled", "checkbox": {"equals": True}},
        {"property": "Status", "select": {"does_not_equal": "Archived"}},
    ]
}

# Sort by Priority then Next Check (best-effort)
preferred_sorts = [
    {"property": "Priority", "direction": "ascending"},
    {"property": "Next Check", "direction": "ascending"},
]

print("Fetching active monitoring targets...\n")

prop_types = DB_PROP_TYPES.get("monitoring_targets", {}) or {}

try:
    raw_targets = notion_query_data_source_all(
        MONITORING_DS_ID,
        filter_obj=preferred_filter,
        sorts=preferred_sorts,
        page_size=100,
        max_pages=50
    )
except requests.HTTPError as e:
    print("⚠ Server-side filter/sort was rejected; falling back to unfiltered query + local filtering.")
    print(f"  Reason: {e}")
    raw_targets = notion_query_data_source_all(
        MONITORING_DS_ID,
        filter_obj=None,
        sorts=None,
        page_size=100,
        max_pages=50
    )

print(f"Retrieved {len(raw_targets)} pages (pre local-filter) from Monitoring Targets data source")

records: List[Dict[str, Any]] = []

def _get(pname: str, page_props: Dict[str, Any]) -> Any:
    return extract_property(page_props.get(pname), prop_types.get(pname))

for page in raw_targets:
    page_id = normalize_uuid(page.get("id"))
    props = page.get("properties", {}) or {}

    enabled = _get("Enabled", props)
    status = _get("Status", props)
    status_norm = (status or "").strip().lower()

    # Defensive local filter
    if enabled is not True:
        continue
    if status_norm == "archived":
        continue

    rec = {
        "page_id": page_id,
        "name": _get("Name", props) or "Untitled",
        "type": _get("Type", props) or "Unknown",
        "status": status or "Active",
        "priority": _get("Priority", props) or "Medium",
        "cadence": _get("Cadence", props) or "Weekly",
        "search_keywords": _get("Search Keywords", props) or "",
        "source_urls": _get("Source URLs", props) or "",
        "last_checked": _get("Last Checked", props),
        "next_check": _get("Next Check", props),
        "source_type": _get("Source Type", props) or "",
        "last_error": _get("Last Error", props) or "",
        "error_count": _get("Error Count", props),
    }
    records.append(rec)

targets_df = pd.DataFrame(records)

print(f"Filtered to {len(targets_df)} active, enabled targets")

if targets_df.empty:
    print("⚠ No active monitoring targets found after filtering.")
else:
    # Basic summaries
    print("\nActive Monitoring Targets Summary:")
    print(f"  Total: {len(targets_df)}")

    for col in ["type", "priority", "cadence"]:
        if col in targets_df.columns:
            print(f"\nBreakdown by {col}:")
            print(targets_df[col].fillna("Unknown").value_counts().to_string())

    print("\nSample targets (first 5):")
    print(targets_df[["name", "type", "priority", "cadence"]].head(5).to_string(index=False))

print("\n✓ Monitoring targets loaded successfully")


Fetching active monitoring targets...

Retrieved 32 pages (pre local-filter) from Monitoring Targets data source
Filtered to 32 active, enabled targets

Active Monitoring Targets Summary:
  Total: 32

Breakdown by type:
type
POLICY     16
VC          9
PEOPLE      6
STARTUP     1

Breakdown by priority:
priority
High      23
Medium     9

Breakdown by cadence:
cadence
DAILY     17
WEEKLY    15

Sample targets (first 5):
                               name   type priority cadence
         UK Research and Innovation POLICY     High   DAILY
                          GRANTSGOV POLICY     High   DAILY
                          NIH_GUIDE POLICY     High   DAILY
                         内閣府 科学技術政策 POLICY     High   DAILY
Japan Science and Technology Agency POLICY     High   DAILY

✓ Monitoring targets loaded successfully


In [19]:
# ============================================================
# Cell 06 — Fetch Events from past N days with target relations (data_sources query only)
# ============================================================
# Overview:
#   Fetch events from the Events DB for the last N days and keep the Target relation
#   to join with monitoring targets. Uses POST /v1/data_sources/{data_source_id}/query only.
#
# Inputs / Outputs:
#   Inputs:  RESOLVED_DB["events"]["data_source_id"], DB_PROP_TYPES, extract_property(),
#            notion_query_data_source_all() (from Cell 05)
#   Outputs: events_df (pandas DataFrame), raw_events (list of Notion pages)
#
# Notes:
#   - Server-side date filtering is best-effort; always apply local filtering.
#   - Uses JST as the reference timezone for the lookback window.
#

EVENTS_DS_ID = RESOLVED_DB["events"]["data_source_id"]
events_prop_types = DB_PROP_TYPES.get("events", {}) or {}

# --- Lookback configuration ---
DAYS_LOOKBACK = 7  # change if needed

now_jst = datetime.now(tz=JST)
start_jst = (now_jst - timedelta(days=DAYS_LOOKBACK))

# For logs
print(f"Fetching Events from past {DAYS_LOOKBACK} days (JST)...")
print(f"  Window: {start_jst.isoformat()}  →  {now_jst.isoformat()}\n")

def _get_event(pname: str, props: Dict[str, Any]) -> Any:
    return extract_property(props.get(pname), events_prop_types.get(pname))

def _parse_iso(dt_str: Optional[str]) -> Optional[datetime]:
    """
    Parse Notion date 'start' into a timezone-aware datetime if possible.
    - If 'YYYY-MM-DD' => treat as JST midnight.
    - If ISO datetime => keep offset if present, else assume JST.
    """
    if not dt_str:
        return None
    s = str(dt_str).strip()
    try:
        if "T" in s:
            # handle trailing Z
            s2 = s.replace("Z", "+00:00")
            dt = datetime.fromisoformat(s2)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=JST)
            return dt.astimezone(JST)
        # date-only
        d = datetime.strptime(s, "%Y-%m-%d")
        return d.replace(tzinfo=JST)
    except Exception:
        return None

# --- Best-effort server-side filter by Date >= start_jst (may fail depending on Notion filter support) ---
# We try to filter on "Date" property which is required in EVENTS_SCHEMA.
preferred_filter = {
    "property": "Date",
    "date": {"on_or_after": start_jst.date().isoformat()},
}

# Sort by Date desc (best-effort)
preferred_sorts = [{"property": "Date", "direction": "descending"}]

try:
    raw_events = notion_query_data_source_all(
        EVENTS_DS_ID,
        filter_obj=preferred_filter,
        sorts=preferred_sorts,
        page_size=100,
        max_pages=80
    )
except requests.HTTPError as e:
    print("⚠ Server-side date filter/sort was rejected; falling back to unfiltered query + local filtering.")
    print(f"  Reason: {e}")
    raw_events = notion_query_data_source_all(
        EVENTS_DS_ID,
        filter_obj=None,
        sorts=None,
        page_size=100,
        max_pages=80
    )

print(f"Retrieved {len(raw_events)} pages (pre local-filter) from Events data source")

events_records: List[Dict[str, Any]] = []

for page in raw_events:
    page_id = normalize_uuid(page.get("id"))
    props = page.get("properties", {}) or {}

    # Required props per your EVENTS_SCHEMA
    name = _get_event("Name", props) or "Untitled"
    date_start = _get_event("Date", props)  # ISO start string
    detected_at = _get_event("Detected At", props)
    event_type = _get_event("Event Type", props) or "Unknown"
    source_url = _get_event("Source URL", props) or ""
    source = _get_event("Source", props) or ""
    summary = _get_event("Summary", props) or ""
    confidence = _get_event("Confidence", props)  # number
    dedup_key = _get_event("Dedup Key", props) or ""
    status = _get_event("Status", props) or ""
    action_needed = _get_event("Action Needed", props)
    target_rel = _get_event("Target", props) or []  # relation -> list of page IDs

    dt = _parse_iso(date_start) or _parse_iso(detected_at)

    # Local filter: keep only last N days (prefer Date, fallback Detected At)
    if not dt:
        continue
    if dt < start_jst or dt > now_jst:
        continue

    events_records.append({
        "page_id": page_id,
        "name": name,
        "date": date_start,
        "detected_at": detected_at,
        "dt_jst": dt,
        "event_type": event_type,
        "source": source,
        "source_url": source_url,
        "summary": summary,
        "confidence": confidence,
        "dedup_key": dedup_key,
        "status": status,
        "action_needed": bool(action_needed) if action_needed is not None else False,
        "target_ids": [normalize_uuid(t) for t in target_rel if t],
        "target_count": len(target_rel),
    })

events_df = pd.DataFrame(events_records)

print(f"Filtered to {len(events_df)} events within past {DAYS_LOOKBACK} days (local filter)")

if events_df.empty:
    print("\n⚠ No events found in the specified date range after filtering.")
else:
    print("\nRecent Events Summary:")
    print(f"  Total events: {len(events_df)}")
    print(f"  Events with target relations: {(events_df['target_count'] > 0).sum()}")
    print(f"  Events without target relations: {(events_df['target_count'] == 0).sum()}")

    if "event_type" in events_df.columns:
        print("\nBreakdown by Event Type:")
        print(events_df["event_type"].fillna("Unknown").value_counts().to_string())

    # Confidence is numeric; show bins only if present
    if events_df["confidence"].notna().any():
        print("\nConfidence (numeric) quick stats:")
        print(events_df["confidence"].describe().to_string())

    sample = events_df[events_df["target_count"] > 0].head(5)
    if not sample.empty:
        print("\nSample events with target relations (first 5):")
        print(sample[["name", "event_type", "action_needed", "confidence", "target_count"]].to_string(index=False))

print("\n✓ Events loaded successfully")


Fetching Events from past 7 days (JST)...
  Window: 2026-02-01T11:02:15.292425+09:00  →  2026-02-08T11:02:15.292425+09:00

Retrieved 69 pages (pre local-filter) from Events data source
Filtered to 63 events within past 7 days (local filter)

Recent Events Summary:
  Total events: 63
  Events with target relations: 63
  Events without target relations: 0

Breakdown by Event Type:
event_type
PEOPLE    46
POLICY    14
VC         3

Confidence (numeric) quick stats:
count    6.300000e+01
mean     6.000000e-01
std      2.238281e-16
min      6.000000e-01
25%      6.000000e-01
50%      6.000000e-01
75%      6.000000e-01
max      6.000000e-01

Sample events with target relations (first 5):
                                                                                                                                                                      name event_type  action_needed  confidence  target_count
                    ‘I wonder why Anthropic would go for something so clearly dishones

In [20]:
# ============================================================
# Cell 07 — Aggregate Events per target and compute signal metrics
# ============================================================
# Overview:
#   Join events_df to targets_df via relation IDs and compute per-target weekly metrics:
#     - number_of_events (7d)
#     - share_of_action_needed
#     - average_confidence (numeric)
#     - duplicate/noise proxy via Dedup Key repetition
#     - freshness proxy via most recent event datetime (JST)
#
# Inputs / Outputs:
#   Inputs:  targets_df (Cell 05), events_df (Cell 06)
#   Outputs: metrics_df (per-target metrics), events_by_target_df (event rows exploded by target)
#
# Notes:
#   - Works even if some events have no target relations (they are excluded from target aggregation).
#   - Confidence is numeric in EVENTS_SCHEMA; NaNs are handled safely.
#

print("Aggregating events per target and computing signal metrics...\n")

# --- Defaults for thresholds (if not defined elsewhere) ---
MIN_SIGNAL_THRESHOLD = globals().get("MIN_SIGNAL_THRESHOLD", 0.35)
HIGH_NOISE_THRESHOLD = globals().get("HIGH_NOISE_THRESHOLD", 0.65)

if targets_df is None or targets_df.empty:
    raise RuntimeError("targets_df is empty. Run Cell 05 first.")

if events_df is None or events_df.empty:
    print("⚠ events_df is empty. Metrics will be computed with zero events for all targets.")
    # Create empty skeleton metrics_df and exit gracefully
    metrics_df = targets_df.copy()
    metrics_df["number_of_events"] = 0
    metrics_df["share_action_needed"] = 0.0
    metrics_df["avg_confidence"] = np.nan
    metrics_df["dedup_dup_rate"] = 0.0
    metrics_df["last_event_dt_jst"] = pd.NaT
    metrics_df["days_since_last_event"] = np.nan
    metrics_df["signal_score"] = 0.0
    metrics_df["noise_score"] = 0.0
    print("✓ Signal metrics computed successfully (no events)")
else:
    # --- Explode events by target relation ---
    # Keep only events that have at least one target relation
    ev = events_df.copy()
    ev["target_ids"] = ev["target_ids"].apply(lambda x: x if isinstance(x, list) else [])
    ev_rel = ev.explode("target_ids", ignore_index=True)
    ev_rel = ev_rel[ev_rel["target_ids"].notna() & (ev_rel["target_ids"] != "")]
    ev_rel = ev_rel.rename(columns={"target_ids": "target_id"})

    # Normalize IDs for safe joins
    ev_rel["target_id"] = ev_rel["target_id"].apply(normalize_uuid)
    targets_norm = targets_df.copy()
    targets_norm["page_id"] = targets_norm["page_id"].apply(normalize_uuid)

    # Join event rows with target attributes (left join keeps event rows only where target exists)
    events_by_target_df = ev_rel.merge(
        targets_norm,
        left_on="target_id",
        right_on="page_id",
        how="inner",
        suffixes=("_event", "_target"),
    )

    mapped_targets = events_by_target_df["target_id"].nunique()
    print(f"Mapped events to {mapped_targets} targets (targets in targets_df: {targets_df['page_id'].nunique()})")

    # --- Helper: dedup duplicate rate per target ---
    # dup_rate = 1 - (unique dedup keys / total events), using non-empty keys; if none, 0
    def _dedup_dup_rate_from_series(keys: pd.Series) -> float:
        keys = keys.fillna("").astype(str)
        keys = keys[keys.str.strip() != ""]
        if len(keys) == 0:
            return 0.0
        return float(1.0 - (keys.nunique() / len(keys)))

    # Ensure dt_jst is datetime
    # Ensure dt_jst is pandas datetime with JST tz (robust for tz-aware dtypes)
    # - If already tz-aware, normalize to JST
    # - If naive/strings, parse then localize/convert
    events_by_target_df["dt_jst"] = pd.to_datetime(events_by_target_df["dt_jst"], errors="coerce")
    
    # If dt_jst is tz-naive, localize to JST; if tz-aware, convert to JST
    if getattr(events_by_target_df["dt_jst"].dt, "tz", None) is None:
        events_by_target_df["dt_jst"] = events_by_target_df["dt_jst"].dt.tz_localize(JST)
    else:
        events_by_target_df["dt_jst"] = events_by_target_df["dt_jst"].dt.tz_convert(JST)


    # Compute per-target aggregations
    g = events_by_target_df.groupby("target_id", dropna=False)

    agg = pd.DataFrame({
        "number_of_events": g.size(),
        "share_action_needed": g["action_needed"].mean(),
        "avg_confidence": g["confidence"].mean(),  # numeric mean (NaNs auto-handled)
        "dedup_dup_rate": g["dedup_key"].apply(_dedup_dup_rate_from_series),
        "last_event_dt_jst": g["dt_jst"].max(),
    }).reset_index()

    # Days since last event (JST)
    now_jst_ts = pd.Timestamp(now_jst)
    agg["days_since_last_event"] = (now_jst_ts - pd.to_datetime(agg["last_event_dt_jst"])).dt.total_seconds() / 86400.0

    # --- Simple derived scores (transparent & tweakable) ---
    # Signal: volume + action-needed share + confidence (if present)
    # Noise: dedup duplicate rate + low confidence proxy (if confidence present)
    # Normalize volume to 0..1 using a cap of 10 events/week
    agg["volume_score"] = (agg["number_of_events"] / 10.0).clip(0, 1)

    # Confidence normalization: assume confidence is 0..1 or 0..100; auto-scale if >1.5
    conf = agg["avg_confidence"].copy()
    conf_scaled = conf
    conf_scaled = np.where(conf_scaled > 1.5, conf_scaled / 100.0, conf_scaled)  # 0..100 -> 0..1
    conf_scaled = pd.Series(conf_scaled).clip(0, 1)
    agg["confidence_score"] = conf_scaled

    # Recency score: decay by days since last event (0..1)
    # score = exp(-days/7)
    agg["recency_score"] = np.exp(-agg["days_since_last_event"].fillna(999) / 7.0)

    agg["signal_score"] = (
        0.45 * agg["volume_score"] +
        0.35 * agg["share_action_needed"].fillna(0) +
        0.20 * agg["confidence_score"].fillna(0)
    ).clip(0, 1)

    # Noise score: duplicates + inverse confidence (when available)
    agg["low_confidence_proxy"] = (1.0 - agg["confidence_score"].fillna(0.5)).clip(0, 1)
    agg["noise_score"] = (
        0.60 * agg["dedup_dup_rate"].fillna(0) +
        0.40 * agg["low_confidence_proxy"]
    ).clip(0, 1)

    # Merge back to targets_df so every target is present (targets with 0 events get defaults)
    metrics_df = targets_norm.merge(
        agg,
        left_on="page_id",
        right_on="target_id",
        how="left",
    )

    # Fill targets with no events
    metrics_df["number_of_events"] = metrics_df["number_of_events"].fillna(0).astype(int)
    metrics_df["share_action_needed"] = metrics_df["share_action_needed"].fillna(0.0)
    metrics_df["dedup_dup_rate"] = metrics_df["dedup_dup_rate"].fillna(0.0)
    metrics_df["signal_score"] = metrics_df["signal_score"].fillna(0.0)
    metrics_df["noise_score"] = metrics_df["noise_score"].fillna(0.0)

    # Keep clean columns
    # (drop helper join column if present)
    if "target_id" in metrics_df.columns:
        metrics_df = metrics_df.drop(columns=["target_id"], errors="ignore")

    # --- Summary prints ---
    print("\nSignal Metrics Summary:")
    print(f"  Targets analyzed: {len(metrics_df)}")
    print(f"  Targets with events: {(metrics_df['number_of_events'] > 0).sum()}")
    print(f"  Targets with no events: {(metrics_df['number_of_events'] == 0).sum()}")

    print("\nSignal score statistics:")
    print(metrics_df["signal_score"].describe().to_string())

    print("\nNoise score statistics:")
    print(metrics_df["noise_score"].describe().to_string())

    low_signal = metrics_df[metrics_df["signal_score"] < MIN_SIGNAL_THRESHOLD]
    high_noise = metrics_df[metrics_df["noise_score"] > HIGH_NOISE_THRESHOLD]
    print(f"\nLow-signal targets (signal_score < {MIN_SIGNAL_THRESHOLD}): {len(low_signal)}")
    print(f"High-noise targets (noise_score > {HIGH_NOISE_THRESHOLD}): {len(high_noise)}")

    print("\nTop 10 targets by signal_score:")
    cols_show = ["name", "type", "priority", "cadence", "number_of_events", "signal_score", "noise_score"]
    print(metrics_df.sort_values("signal_score", ascending=False)[cols_show].head(10).to_string(index=False))

    print("\nBottom 10 targets by signal_score (review candidates):")
    print(metrics_df.sort_values("signal_score", ascending=True)[cols_show].head(10).to_string(index=False))

    print("\n✓ Signal metrics computed successfully")


Aggregating events per target and computing signal metrics...

Mapped events to 9 targets (targets in targets_df: 32)

Signal Metrics Summary:
  Targets analyzed: 32
  Targets with events: 9
  Targets with no events: 23

Signal score statistics:
count    32.000000
mean      0.184219
std       0.310655
min       0.000000
25%       0.000000
50%       0.000000
75%       0.515000
max       0.920000

Noise score statistics:
count    32.000000
mean      0.045000
std       0.073089
min       0.000000
25%       0.000000
50%       0.000000
75%       0.160000
max       0.160000

Low-signal targets (signal_score < 0.35): 23
High-noise targets (noise_score > 0.7): 0

Top 10 targets by signal_score:
                                               name   type priority cadence  number_of_events  signal_score  noise_score
                                       Jensen Huang PEOPLE     High  WEEKLY                24         0.920         0.16
                                         Sam Altman PEOPLE    

In [21]:
# ============================================================
# Cell 08 — LLM-based target effectiveness evaluation
# ============================================================
# Overview:
#   Use an LLM to qualitatively evaluate monitoring target effectiveness based on:
#     - computed weekly metrics (signal_score, noise_score, counts)
#     - a compact summary of recent events linked to the target
#   Produces evaluation_df with effectiveness_score, refined_noise_ratio, rationale.
#
# Inputs / Outputs:
#   Inputs:  metrics_df (Cell 07), events_by_target_df (Cell 07, if events exist),
#            OPENAI_API_KEY (env), LLM_MODEL/LLM_TEMPERATURE (Cell 01)
#   Outputs: evaluation_df (pandas DataFrame)
#
# Notes:
#   - To control cost, evaluates a subset by default (review candidates + top signals).
#   - Robust JSON parsing with fallbacks.
#   - If OPENAI_API_KEY is missing, this cell will skip LLM evaluation gracefully.
#

import json

print("Evaluating target effectiveness using LLM...\n")

# --- Guard: if no API key, skip gracefully ---
if not OPENAI_API_KEY:
    print("⚠ OPENAI_API_KEY is not set. Skipping LLM evaluation.")
    evaluation_df = pd.DataFrame()
else:
    # Lazy import + client init inside this cell
    try:
        from openai import OpenAI
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
    except Exception as e:
        print(f"⚠ Failed to initialize OpenAI client. Skipping LLM evaluation. Reason: {e}")
        evaluation_df = pd.DataFrame()

# --- Utility: safe JSON extraction ---
def _extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    """
    Try to extract the first JSON object from text and parse it.
    Returns dict or None.
    """
    if not text or not isinstance(text, str):
        return None
    s = text.strip()
    # Fast path: pure JSON
    try:
        return json.loads(s)
    except Exception:
        pass
    # Try to locate first {...}
    start = s.find("{")
    end = s.rfind("}")
    if start >= 0 and end > start:
        snippet = s[start:end+1]
        try:
            return json.loads(snippet)
        except Exception:
            return None
    return None

# --- Build per-target event summaries from events_by_target_df ---
def build_event_summary_for_target(target_id: str, max_events: int = 8) -> str:
    """
    Create a compact bullet list of recent events for the target.
    Requires events_by_target_df. If not available, returns a fallback string.
    """
    if "events_by_target_df" not in globals() or events_by_target_df is None or events_by_target_df.empty:
        return "No per-target event rows available (events_by_target_df missing/empty)."

    df = events_by_target_df[events_by_target_df["target_id"] == target_id].copy()
    if df.empty:
        return "No events linked to this target in the past window."

    # Sort by dt_jst desc if present
    if "dt_jst" in df.columns:
        df = df.sort_values("dt_jst", ascending=False)

    lines = []
    for _, r in df.head(max_events).iterrows():
        name = str(r.get("name_event", r.get("name", "Untitled"))).strip()
        etype = str(r.get("event_type", "Unknown")).strip()
        conf = r.get("confidence", None)
        conf_txt = "NA"
        if conf is not None and not (isinstance(conf, float) and np.isnan(conf)):
            conf_txt = f"{conf}"
        action = "ACTION" if bool(r.get("action_needed", False)) else "—"
        lines.append(f"- {name} | {etype} | conf={conf_txt} | {action}")
    if len(df) > max_events:
        lines.append(f"- ... and {len(df) - max_events} more events")
    return "\n".join(lines)

# --- LLM call ---
def evaluate_target_effectiveness_llm(
    *,
    target_name: str,
    target_type: str,
    priority: str,
    cadence: str,
    number_of_events: int,
    signal_score: float,
    noise_score: float,
    share_action_needed: float,
    avg_confidence: Optional[float],
    dedup_dup_rate: float,
    days_since_last_event: Optional[float],
    event_summary: str,
    model: str,
    temperature: float
) -> Dict[str, Any]:
    """
    Returns dict:
      effectiveness_score (0..1),
      refined_noise_ratio (0..1),
      recommendation (one of keep / tune / deprioritize / disable),
      rationale (short)
    """
    prompt = f"""
You are an expert analyst evaluating the effectiveness of a monitoring target in a weekly review system.

Target: {target_name}
Type: {target_type}
Priority: {priority}
Cadence: {cadence}

Computed weekly metrics (0..1 unless noted):
- number_of_events (7d): {number_of_events}
- signal_score: {signal_score:.3f}
- noise_score: {noise_score:.3f}
- share_action_needed: {share_action_needed:.3f}
- avg_confidence: {avg_confidence if avg_confidence is not None else "NA"}  (numeric)
- dedup_dup_rate: {dedup_dup_rate:.3f}  (0=all unique, 1=all duplicates)
- days_since_last_event: {days_since_last_event if days_since_last_event is not None else "NA"}

Recent linked events (most recent first):
{event_summary}

Task:
1) Provide an effectiveness_score (0..1): overall value of keeping this target in the watchlist.
2) Provide refined_noise_ratio (0..1): % of low-value/noisy items.
3) Provide a recommendation: one of ["keep", "tune", "deprioritize", "disable"].
4) Provide a short rationale (2-4 sentences) referencing the metrics and event patterns.

Respond as STRICT JSON only:
{{
  "effectiveness_score": 0.0,
  "refined_noise_ratio": 0.0,
  "recommendation": "keep|tune|deprioritize|disable",
  "rationale": "..."
}}
""".strip()

    try:
        resp = openai_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a careful analyst. Output strict JSON only."},
                {"role": "user", "content": prompt},
            ],
            temperature=temperature,
            max_tokens=400,
        )
        text = (resp.choices[0].message.content or "").strip()
        obj = _extract_json_object(text)

        if not isinstance(obj, dict):
            raise ValueError(f"Non-JSON response: {text[:200]}")

        eff = float(obj.get("effectiveness_score", signal_score))
        rn = float(obj.get("refined_noise_ratio", noise_score))
        rec = str(obj.get("recommendation", "tune")).strip().lower()
        rat = str(obj.get("rationale", "")).strip() or "No rationale provided."

        # Clamp
        eff = max(0.0, min(1.0, eff))
        rn = max(0.0, min(1.0, rn))
        if rec not in {"keep", "tune", "deprioritize", "disable"}:
            rec = "tune"

        return {
            "effectiveness_score": eff,
            "refined_noise_ratio": rn,
            "recommendation": rec,
            "rationale": rat,
            "raw": text,
        }
    except Exception as e:
        # Fallback: rely on computed metrics
        return {
            "effectiveness_score": float(max(0.0, min(1.0, signal_score))),
            "refined_noise_ratio": float(max(0.0, min(1.0, noise_score))),
            "recommendation": "tune" if number_of_events > 0 else "deprioritize",
            "rationale": f"LLM evaluation failed; fallback to computed metrics. Error: {e}",
            "raw": None,
        }

# --- Choose which targets to evaluate (cost control) ---
# Default: evaluate review candidates + top signals (up to 16 total)
MAX_EVAL = 16

if "evaluation_df" in globals() and isinstance(evaluation_df, pd.DataFrame) and evaluation_df.empty and not OPENAI_API_KEY:
    # already skipped
    pass
else:
    # Ensure required columns exist
    required_cols = ["page_id", "name", "type", "priority", "cadence",
                     "number_of_events", "signal_score", "noise_score",
                     "share_action_needed", "avg_confidence", "dedup_dup_rate",
                     "days_since_last_event"]
    missing_cols = [c for c in required_cols if c not in metrics_df.columns]
    if missing_cols:
        raise RuntimeError(f"metrics_df missing required columns: {missing_cols}")

    # Pick candidates: bottom by signal OR top by noise OR top by signal
    bottom_signal = metrics_df.sort_values("signal_score", ascending=True).head(6)
    top_noise = metrics_df.sort_values("noise_score", ascending=False).head(6)
    top_signal = metrics_df.sort_values("signal_score", ascending=False).head(6)

    candidate_df = pd.concat([bottom_signal, top_noise, top_signal], ignore_index=True).drop_duplicates(subset=["page_id"])
    candidate_df = candidate_df.head(MAX_EVAL).reset_index(drop=True)
    candidate_df = metrics_df

    print(f"Evaluating {len(candidate_df)} targets via LLM (subset, MAX_EVAL={MAX_EVAL})\n")

    rows = []
    for i, r in candidate_df.iterrows():
        tid = r["page_id"]
        tname = r["name"]
        print(f"  [{i+1}/{len(candidate_df)}] {tname}")

        ev_summary = build_event_summary_for_target(tid, max_events=8)

        out = evaluate_target_effectiveness_llm(
            target_name=tname,
            target_type=str(r.get("type", "")),
            priority=str(r.get("priority", "")),
            cadence=str(r.get("cadence", "")),
            number_of_events=int(r.get("number_of_events", 0)),
            signal_score=float(r.get("signal_score", 0.0)),
            noise_score=float(r.get("noise_score", 0.0)),
            share_action_needed=float(r.get("share_action_needed", 0.0)),
            avg_confidence=(None if pd.isna(r.get("avg_confidence")) else float(r.get("avg_confidence"))),
            dedup_dup_rate=float(r.get("dedup_dup_rate", 0.0)),
            days_since_last_event=(None if pd.isna(r.get("days_since_last_event")) else float(r.get("days_since_last_event"))),
            event_summary=ev_summary,
            model=LLM_MODEL,
            temperature=LLM_TEMPERATURE,
        )

        rows.append({
            "page_id": tid,
            "name": tname,
            "type": r.get("type"),
            "priority": r.get("priority"),
            "cadence": r.get("cadence"),
            "number_of_events": r.get("number_of_events"),
            "computed_signal_score": r.get("signal_score"),
            "computed_noise_score": r.get("noise_score"),
            "llm_effectiveness_score": out["effectiveness_score"],
            "llm_refined_noise_ratio": out["refined_noise_ratio"],
            "llm_recommendation": out["recommendation"],
            "llm_rationale": out["rationale"],
        })

    evaluation_df = pd.DataFrame(rows)

    print(f"\n✓ LLM evaluation complete for {len(evaluation_df)} targets")

    if not evaluation_df.empty:
        print("\nLLM Effectiveness Scores:")
        print(evaluation_df["llm_effectiveness_score"].describe().to_string())

        print("\nTop targets by LLM effectiveness:")
        print(evaluation_df.sort_values("llm_effectiveness_score", ascending=False)[
            ["name", "type", "number_of_events", "llm_effectiveness_score", "llm_recommendation"]
        ].head(10).to_string(index=False))

        print("\nBottom targets by LLM effectiveness (review candidates):")
        print(evaluation_df.sort_values("llm_effectiveness_score", ascending=True)[
            ["name", "type", "number_of_events", "llm_effectiveness_score", "llm_recommendation", "llm_rationale"]
        ].head(10).to_string(index=False))

print("\n✓ Target effectiveness evaluation complete")


Evaluating target effectiveness using LLM...

Evaluating 32 targets via LLM (subset, MAX_EVAL=16)

  [1/32] UK Research and Innovation
  [2/32] GRANTSGOV
  [3/32] NIH_GUIDE
  [4/32] 内閣府 科学技術政策
  [5/32] Japan Science and Technology Agency
  [6/32] NEDO
  [7/32] AMED
  [8/32] perplexity
  [9/32] Reid Hoffman
  [10/32] Masayoshi Son
  [11/32] Jensen Huang
  [12/32] Sam Altman
  [13/32] Peter Thiel
  [14/32] Eduardo Saverin
  [15/32] UTEC
  [16/32] Global Brain
  [17/32] Temasek
  [18/32] Lightspeed India
  [19/32] Lightspeed (US)
  [20/32] a16z (Andreessen Horowitz)
  [21/32] Peak XV (Sequoia India)
  [22/32] Sequoia Capital
  [23/32] B Capital
  [24/32] OECD Science, Technology and Innovation
  [25/32] Agence nationale de la recherche
  [26/32] German Research Foundation
  [27/32] Medicines and Healthcare products Regulatory Agency
  [28/32] European Commission Digital Strategy
  [29/32] European Medicines Agency
  [30/32] OSTP
  [31/32] NIH
  [32/32] NIH_NEXUS

✓ LLM evaluation complete

In [22]:
# ============================================================
# Cell 09 — Identify targets for priority adjustment or removal
# ============================================================
# Overview:
#   Combine computed metrics (metrics_df) with optional LLM evaluation (evaluation_df)
#   and propose actions:
#     - disable / deprioritize / keep / tune
#     - priority up/down recommendations
#
# Inputs / Outputs:
#   Inputs:  metrics_df, (optional) evaluation_df
#   Outputs: actions_df, removal_df, downgrade_df, upgrade_df
#
# Notes:
#   - Works even if evaluation_df is missing/empty (LLM skipped).
#   - Uses conservative, explainable rules; thresholds are tweakable.
#

print("Identifying targets for priority adjustment or removal...\n")

# --- Thresholds (tweak as needed) ---
REMOVAL_EFFECTIVENESS_THRESHOLD = 0.25
DOWNGRADE_EFFECTIVENESS_THRESHOLD = 0.40
UPGRADE_EFFECTIVENESS_THRESHOLD = 0.75
HIGH_NOISE_REMOVAL_THRESHOLD = 0.80

# Also use computed metrics as fallback
LOW_SIGNAL_THRESHOLD = 0.30
HIGH_NOISE_THRESHOLD = 0.70
NO_EVENT_DAYS_THRESHOLD = 14  # if no events for 2 weeks, candidate to deprioritize

# Priority order (best-effort; unknown values default to Medium index)
priority_order = ["Low", "Medium", "High", "Critical"]

def get_priority_index(priority: str) -> int:
    p = (priority or "").strip()
    if p in priority_order:
        return priority_order.index(p)
    return priority_order.index("Medium")

def adjust_priority(current_priority: str, direction: str) -> str:
    idx = get_priority_index(current_priority)
    if direction == "up" and idx < len(priority_order) - 1:
        return priority_order[idx + 1]
    if direction == "down" and idx > 0:
        return priority_order[idx - 1]
    return current_priority

# --- Build unified table (metrics + optional LLM) ---
base = metrics_df.copy()

# If evaluation_df exists, merge it; else create empty columns
if "evaluation_df" in globals() and isinstance(evaluation_df, pd.DataFrame) and not evaluation_df.empty:
    merged = base.merge(
        evaluation_df[[
            "page_id",
            "llm_effectiveness_score",
            "llm_refined_noise_ratio",
            "llm_recommendation",
            "llm_rationale",
        ]],
        on="page_id",
        how="left",
    )
else:
    merged = base.copy()
    merged["llm_effectiveness_score"] = np.nan
    merged["llm_refined_noise_ratio"] = np.nan
    merged["llm_recommendation"] = ""
    merged["llm_rationale"] = ""

# Choose primary scores:
# - effectiveness: LLM if available else signal_score
# - noise: LLM refined if available else noise_score
merged["effectiveness"] = merged["llm_effectiveness_score"]
merged["effectiveness"] = merged["effectiveness"].where(~merged["effectiveness"].isna(), merged["signal_score"])
merged["noise_ratio"] = merged["llm_refined_noise_ratio"]
merged["noise_ratio"] = merged["noise_ratio"].where(~merged["noise_ratio"].isna(), merged["noise_score"])

# Ensure numeric
for c in ["effectiveness", "noise_ratio", "signal_score", "noise_score", "share_action_needed", "dedup_dup_rate"]:
    if c in merged.columns:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

# --- Decision rules ---
removal_candidates = []
downgrade_candidates = []
upgrade_candidates = []
actions = []

for _, r in merged.iterrows():
    tid = r["page_id"]
    name = r.get("name", "Untitled")
    ttype = r.get("type", "")
    priority = r.get("priority", "Medium")
    cadence = r.get("cadence", "")
    n_events = int(r.get("number_of_events", 0) or 0)
    eff = float(r.get("effectiveness", 0.0) or 0.0)
    noise = float(r.get("noise_ratio", 0.0) or 0.0)
    sig = float(r.get("signal_score", 0.0) or 0.0)
    days_since = r.get("days_since_last_event", np.nan)
    days_since = None if pd.isna(days_since) else float(days_since)
    llm_rec = (r.get("llm_recommendation", "") or "").strip().lower()
    llm_rat = (r.get("llm_rationale", "") or "").strip()

    # Base action + reason
    action = "keep"
    reason_parts = []

    # Rule A: Removal/disable candidates
    if eff < REMOVAL_EFFECTIVENESS_THRESHOLD:
        action = "disable"
        reason_parts.append(f"very low effectiveness ({eff:.2f})")
    elif noise > HIGH_NOISE_REMOVAL_THRESHOLD and n_events < 3:
        action = "disable"
        reason_parts.append(f"high noise ({noise:.2f}) with few events ({n_events})")
    elif n_events == 0 and eff < DOWNGRADE_EFFECTIVENESS_THRESHOLD and (days_since is None or days_since >= NO_EVENT_DAYS_THRESHOLD):
        action = "deprioritize"
        reason_parts.append(f"no events recently (days_since_last_event={days_since}) and low effectiveness ({eff:.2f})")

    # Rule B: Tune candidates (not removal)
    if action == "keep":
        if noise > HIGH_NOISE_THRESHOLD:
            action = "tune"
            reason_parts.append(f"high noise ({noise:.2f})")
        elif sig < LOW_SIGNAL_THRESHOLD and n_events > 0:
            action = "tune"
            reason_parts.append(f"low signal ({sig:.2f}) despite events ({n_events})")
        elif n_events == 0 and (days_since is None or days_since >= 7):
            action = "tune"
            reason_parts.append("no events this week")

    # If LLM suggests stronger action, respect it (best-effort)
    if llm_rec in {"disable", "deprioritize"} and action in {"keep", "tune"}:
        action = llm_rec
        reason_parts.append(f"LLM recommends {llm_rec}")

    # Priority adjustment suggestions (skip if disable)
    proposed_priority = priority
    priority_change_reason = ""

    if action != "disable":
        # Upgrade rule: high effectiveness + enough volume OR high action-needed share
        if eff >= UPGRADE_EFFECTIVENESS_THRESHOLD and n_events >= 3:
            proposed_priority = adjust_priority(priority, "up")
            if proposed_priority != priority:
                priority_change_reason = f"upgrade: high effectiveness ({eff:.2f}) with {n_events} events"

        # Downgrade rule: low effectiveness (but not removal) OR sustained no-events
        if proposed_priority == priority:  # avoid conflicting changes
            if eff < DOWNGRADE_EFFECTIVENESS_THRESHOLD and priority != "Low":
                proposed_priority = adjust_priority(priority, "down")
                if proposed_priority != priority:
                    priority_change_reason = f"downgrade: low effectiveness ({eff:.2f})"
            elif (n_events == 0) and (days_since is not None and days_since >= NO_EVENT_DAYS_THRESHOLD) and priority != "Low":
                proposed_priority = adjust_priority(priority, "down")
                if proposed_priority != priority:
                    priority_change_reason = f"downgrade: stale (no events for {days_since:.0f} days)"

    # Record action row
    actions.append({
        "page_id": tid,
        "name": name,
        "type": ttype,
        "current_priority": priority,
        "proposed_priority": proposed_priority,
        "cadence": cadence,
        "number_of_events": n_events,
        "signal_score": sig,
        "noise_ratio": noise,
        "effectiveness": eff,
        "action": action,
        "action_reason": "; ".join(reason_parts) if reason_parts else "",
        "priority_reason": priority_change_reason,
        "llm_recommendation": llm_rec,
        "llm_rationale": llm_rat,
    })

    # Buckets for display
    if action == "disable":
        removal_candidates.append(actions[-1])
    elif proposed_priority != priority and get_priority_index(proposed_priority) < get_priority_index(priority):
        downgrade_candidates.append(actions[-1])
    elif proposed_priority != priority and get_priority_index(proposed_priority) > get_priority_index(priority):
        upgrade_candidates.append(actions[-1])

actions_df = pd.DataFrame(actions)
removal_df = pd.DataFrame(removal_candidates)
downgrade_df = pd.DataFrame(downgrade_candidates)
upgrade_df = pd.DataFrame(upgrade_candidates)

print(f"Identified {len(removal_df)} targets for potential disable/removal")
print(f"Identified {len(downgrade_df)} targets for priority downgrade")
print(f"Identified {len(upgrade_df)} targets for priority upgrade")

# --- Display summaries ---
cols_removal = ["name", "type", "current_priority", "number_of_events", "effectiveness", "noise_ratio", "action_reason"]
cols_change = ["name", "type", "current_priority", "proposed_priority", "number_of_events", "effectiveness", "priority_reason"]

if not removal_df.empty:
    print(f"\nDisable/Removal Candidates ({len(removal_df)}):")
    print(removal_df[cols_removal].head(50).to_string(index=False))
else:
    print("\nNo disable/removal candidates identified")

if not downgrade_df.empty:
    print(f"\nDowngrade Candidates ({len(downgrade_df)}):")
    print(downgrade_df[cols_change].head(50).to_string(index=False))
else:
    print("\nNo downgrade candidates identified")

if not upgrade_df.empty:
    print(f"\nUpgrade Candidates ({len(upgrade_df)}):")
    print(upgrade_df[cols_change].head(50).to_string(index=False))
else:
    print("\nNo upgrade candidates identified")

total_adjustments = len(removal_df) + len(downgrade_df) + len(upgrade_df)
print(f"\n✓ Priority adjustment analysis complete")
print(f"  Total proposed adjustments: {total_adjustments}")
print(f"  Disable/Removal: {len(removal_df)}")
print(f"  Downgrade: {len(downgrade_df)}")
print(f"  Upgrade: {len(upgrade_df)}")


Identifying targets for priority adjustment or removal...

Identified 23 targets for potential disable/removal
Identified 0 targets for priority downgrade
Identified 4 targets for priority upgrade

Disable/Removal Candidates (23):
                                   name    type current_priority  number_of_events  effectiveness  noise_ratio                 action_reason
                              GRANTSGOV  POLICY             High                 0            0.0          0.0 very low effectiveness (0.00)
                             内閣府 科学技術政策  POLICY             High                 0            0.0          0.0 very low effectiveness (0.00)
    Japan Science and Technology Agency  POLICY             High                 0            0.0          0.0 very low effectiveness (0.00)
                                   NEDO  POLICY             High                 0            0.0          0.0 very low effectiveness (0.00)
                                   AMED  POLICY             High

In [23]:
# ============================================================
# Cell 10 — Identify targets for cadence or status change
# ============================================================
# Overview:
#   Propose cadence (monitoring frequency) and optional status changes using:
#     - actions_df (Cell 09): action + proposed_priority
#     - metrics_df: weekly activity metrics
#     - optional evaluation_df: LLM effectiveness / recommendation
#
# Inputs / Outputs:
#   Inputs:  actions_df, metrics_df, (optional) evaluation_df
#   Outputs: cadence_changes_df, status_changes_df
#
# Notes:
#   - Works even if evaluation_df is missing/empty.
#   - Status suggestions are proposals only (no writes).
#

print("Identifying targets for cadence or status change...\n")

# --- Cadence options (edit to match your Notion select options exactly) ---
CADENCE_OPTIONS = ["Daily", "Weekly", "Bi-weekly", "Monthly"]

def get_cadence_index(cadence: str) -> int:
    c = (cadence or "").strip()
    if c in CADENCE_OPTIONS:
        return CADENCE_OPTIONS.index(c)
    # default to Weekly
    return CADENCE_OPTIONS.index("Weekly") if "Weekly" in CADENCE_OPTIONS else 1

def adjust_cadence(current_cadence: str, direction: str) -> str:
    idx = get_cadence_index(current_cadence)
    if direction == "up" and idx > 0:
        return CADENCE_OPTIONS[idx - 1]   # more frequent
    if direction == "down" and idx < len(CADENCE_OPTIONS) - 1:
        return CADENCE_OPTIONS[idx + 1]   # less frequent
    return current_cadence

# --- Join actions_df with optional LLM columns (if not already present) ---
df = actions_df.copy()

# If evaluation_df exists, attach LLM columns for richer rules
if "evaluation_df" in globals() and isinstance(evaluation_df, pd.DataFrame) and not evaluation_df.empty:
    df = df.merge(
        evaluation_df[["page_id", "llm_effectiveness_score", "llm_refined_noise_ratio", "llm_recommendation", "llm_rationale"]],
        on="page_id",
        how="left",
        suffixes=("", "_llm"),
    )
else:
    df["llm_effectiveness_score"] = np.nan
    df["llm_refined_noise_ratio"] = np.nan
    df["llm_recommendation"] = ""
    df["llm_rationale"] = ""

# Pull key computed metrics from metrics_df (ensures we have days_since_last_event etc.)
df = df.merge(
    metrics_df[[
        "page_id", "status", "cadence", "number_of_events", "signal_score", "noise_score",
        "share_action_needed", "dedup_dup_rate", "days_since_last_event"
    ]],
    on="page_id",
    how="left",
    suffixes=("", "_m"),
)

# Primary effectiveness/noise (LLM if available else computed)
eff = df["llm_effectiveness_score"]
df["effectiveness"] = eff.where(~eff.isna(), df["signal_score"])
rn = df["llm_refined_noise_ratio"]
df["noise_ratio"] = rn.where(~rn.isna(), df["noise_score"])

# --- Exclusions: do not propose cadence/status for disables ---
disabled_ids = set(df[df["action"] == "disable"]["page_id"].tolist())

cadence_changes = []
status_changes = []

for _, r in df.iterrows():
    tid = r["page_id"]
    if tid in disabled_ids:
        continue

    name = r.get("name", "Untitled")
    ttype = r.get("type", "")
    current_cadence = (r.get("cadence") or "").strip() or "Weekly"
    current_status = (r.get("status") or "").strip() or "Active"

    n_events = int(r.get("number_of_events", 0) or 0)
    effectiveness = float(r.get("effectiveness", 0.0) or 0.0)
    noise_ratio = float(r.get("noise_ratio", 0.0) or 0.0)
    action_needed_share = float(r.get("share_action_needed", 0.0) or 0.0)
    days_since = r.get("days_since_last_event", np.nan)
    days_since = None if pd.isna(days_since) else float(days_since)

    llm_rec = (r.get("llm_recommendation") or "").strip().lower()
    llm_rat = (r.get("llm_rationale") or "").strip()

    # -------------------------
    # Cadence: increase frequency
    # -------------------------
    # Strong signal: high effectiveness + enough events OR high action-needed share
    if current_cadence != "Daily":
        if (effectiveness >= 0.80 and n_events >= 5) or (effectiveness >= 0.70 and action_needed_share >= 0.40 and n_events >= 3):
            proposed = adjust_cadence(current_cadence, "up")
            if proposed != current_cadence:
                cadence_changes.append({
                    "page_id": tid,
                    "name": name,
                    "type": ttype,
                    "current_cadence": current_cadence,
                    "proposed_cadence": proposed,
                    "reason": f"increase: effectiveness={effectiveness:.2f}, events={n_events}, action_needed_share={action_needed_share:.2f}",
                    "effectiveness": effectiveness,
                    "noise_ratio": noise_ratio,
                    "number_of_events": n_events,
                    "llm_recommendation": llm_rec,
                    "llm_rationale": llm_rat,
                })

    # -------------------------
    # Cadence: decrease frequency
    # -------------------------
    # Weak/quiet: few events + moderate/low effectiveness OR high noise with low volume
    if current_cadence != "Monthly":
        if (n_events <= 1 and effectiveness < 0.60) or (n_events <= 2 and noise_ratio >= 0.65):
            proposed = adjust_cadence(current_cadence, "down")
            if proposed != current_cadence:
                cadence_changes.append({
                    "page_id": tid,
                    "name": name,
                    "type": ttype,
                    "current_cadence": current_cadence,
                    "proposed_cadence": proposed,
                    "reason": f"decrease: effectiveness={effectiveness:.2f}, events={n_events}, noise={noise_ratio:.2f}",
                    "effectiveness": effectiveness,
                    "noise_ratio": noise_ratio,
                    "number_of_events": n_events,
                    "llm_recommendation": llm_rec,
                    "llm_rationale": llm_rat,
                })

    # -------------------------
    # Status change proposals (optional)
    # -------------------------
    # If no events for a while but effectiveness isn't terrible, consider On Hold.
    # If previously On Hold but now active, consider Active.
    # NOTE: Your Notion Status select options may differ; treat as recommendation only.
    if days_since is not None:
        # Propose On Hold if stale but still valuable
        if days_since >= 14 and n_events == 0 and effectiveness >= 0.45 and current_status.lower() not in {"on hold", "paused"}:
            status_changes.append({
                "page_id": tid,
                "name": name,
                "type": ttype,
                "current_status": current_status,
                "proposed_status": "On Hold",
                "reason": f"stale: no events for {days_since:.0f} days but effectiveness={effectiveness:.2f}",
                "effectiveness": effectiveness,
                "noise_ratio": noise_ratio,
                "llm_recommendation": llm_rec,
                "llm_rationale": llm_rat,
            })

        # Propose Active if recent activity and decent effectiveness
        if n_events >= 3 and effectiveness >= 0.60 and current_status.lower() in {"on hold", "paused"}:
            status_changes.append({
                "page_id": tid,
                "name": name,
                "type": ttype,
                "current_status": current_status,
                "proposed_status": "Active",
                "reason": f"reactivate: events={n_events}, effectiveness={effectiveness:.2f}",
                "effectiveness": effectiveness,
                "noise_ratio": noise_ratio,
                "llm_recommendation": llm_rec,
                "llm_rationale": llm_rat,
            })

cadence_changes_df = pd.DataFrame(cadence_changes).drop_duplicates(subset=["page_id", "proposed_cadence"], keep="first")
status_changes_df = pd.DataFrame(status_changes).drop_duplicates(subset=["page_id", "proposed_status"], keep="first")

print(f"Identified {len(cadence_changes_df)} cadence change proposals")
print(f"Identified {len(status_changes_df)} status change proposals")

# --- Display summaries ---
if not cadence_changes_df.empty:
    print("\nCadence change proposals:")
    print(cadence_changes_df[[
        "name", "type", "current_cadence", "proposed_cadence", "number_of_events", "effectiveness", "noise_ratio", "reason"
    ]].head(50).to_string(index=False))
else:
    print("\nNo cadence changes proposed")

if not status_changes_df.empty:
    print("\nStatus change proposals:")
    print(status_changes_df[[
        "name", "type", "current_status", "proposed_status", "effectiveness", "reason"
    ]].head(50).to_string(index=False))
else:
    print("\nNo status changes proposed")

print("\n✓ Cadence and status change analysis complete")


Identifying targets for cadence or status change...

Identified 4 cadence change proposals
Identified 0 status change proposals

Cadence change proposals:
                                               name   type current_cadence proposed_cadence  number_of_events  effectiveness  noise_ratio                                                            reason
                         UK Research and Innovation POLICY           DAILY            Daily                 6           0.85         0.16  increase: effectiveness=0.85, events=6, action_needed_share=1.00
                                       Jensen Huang PEOPLE          WEEKLY            Daily                24           0.85         0.15 increase: effectiveness=0.85, events=24, action_needed_share=1.00
                                         Sam Altman PEOPLE          WEEKLY            Daily                22           0.85         0.18 increase: effectiveness=0.85, events=22, action_needed_share=1.00
Medicines and Healthcare prod

In [37]:
# ============================================================
# Cell 11 — Target tuning suggestions (Keywords / Source URLs)
# ============================================================
# Overview:
#   Propose improvements to monitoring target configurations, focusing on:
#     - Search Keywords
#     - Source URLs
#
#   This cell identifies targets that likely suffer from noise, misalignment,
#   or insufficient coverage, inspects external signals (News / Web),
#   and generates human-reviewable tuning proposals to be written to
#   weekly_target_update DB.
#
#   IMPORTANT:
#   - This cell DOES NOT directly modify Monitoring Targets.
#   - It only generates proposals (diff-style) for weekly review.
#
# Inputs / Outputs:
#   Inputs:
#     - actions_df              (Cell 09)
#     - metrics_df              (Cell 07)
#     - evaluation_df (optional, Cell 08)
#     - Current Target settings:
#         * Search Keywords
#         * Source URLs
#     - External signals (to be added):
#         * NewsAPI (recent titles)
#         * Google CSE (URL/domain distribution)
#
#   Outputs:
#     - target_tuning_proposals : list[dict]
#         Each dict is aligned with weekly_target_update schema:
#           - Target (relation)
#           - Proposal Type
#           - Field
#           - Current Value
#           - Proposed Value
#           - Change Summary
#           - Rationale
#           - Evidence (links / notes)
#
# Notes:
#   - This cell is intentionally split into clearly separated steps.
#   - External API calls and LLM usage should be added incrementally.
#   - All proposals must be REVIEWABLE by humans before application.
#

print("Generating target tuning suggestions (Keywords / Source URLs)...\n")

# ============================================================
# Step 11-1 — Identify targets requiring tuning
# ============================================================
# Criteria examples:
#   - action == "tune" or "deprioritize"
#   - high noise ratio
#   - zero or very low event count
#   - LLM recommendation suggests tuning
#
# Output:
#   tuning_targets_df : subset of targets_df / actions_df
#

# TODO: refine selection logic as tuning heuristics mature
tuning_targets_df = actions_df[
    (actions_df["action"].isin(["tune", "deprioritize"])) |
    (actions_df["noise_ratio"] > 0.6) |
    (actions_df["number_of_events"] == 0)
].copy()

print(f"Identified {len(tuning_targets_df)} targets for potential tuning")

# ============================================================
# Step 11-2 — Load current monitoring configuration
# ============================================================
# For each target:
#   - Load current Search Keywords
#   - Load current Source URLs
#
# These values will populate:
#   - Current Value (Before)
#
# NOTE:
#   - Do NOT mutate these values here.
#   - Treat them as immutable context.
#

# TODO:
#   - Fetch Search Keywords / Source URLs from Monitoring Targets DB
#   - Normalize to lists / strings for comparison
#
# Example placeholder structure:
# current_configs = {
#     target_id: {
#         "search_keywords": [...],
#         "source_urls": [...]
#     }
# }

current_configs = {}

# ============================================================
# Step 11-3A — NewsAPI: fetch recent titles + lightweight term extraction
# ============================================================
# Overview:
#   Fetch recent article titles for a given query (built from current keywords),
#   then extract simple "top terms" from titles to detect topic drift / missing angles.
#
# Notes:
#   - Uses NewsAPI /v2/everything (best-effort).
#   - Stores only lightweight evidence: title, source, url, publishedAt.
#   - Term extraction is intentionally simple (no heavy NLP deps).
#

NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")

def _ensure_newsapi_key() -> None:
    if not NEWSAPI_KEY or not str(NEWSAPI_KEY).strip():
        raise RuntimeError("NEWSAPI_KEY is not set in env.txt (required for NewsAPI).")

def build_newsapi_query_from_keywords(
    keywords_text: str,
    *,
    max_terms: int = 6,
) -> str:
    """
    Build a NewsAPI query string from a target's 'Search Keywords' rich_text field.

    Expected input formats:
      - newline-separated keywords
      - comma-separated keywords
      - a JSON-like string (we do a best-effort extraction)

    Output:
      - a query like: ("OpenAI" OR "ChatGPT" OR "Sam Altman")
    """
    if not keywords_text:
        return ""

    raw = str(keywords_text).strip()

    # Best-effort: if it looks like JSON, try to parse, else fall back to splitting.
    terms: list[str] = []
    try:
        obj = json.loads(raw)
        if isinstance(obj, list):
            terms = [str(x).strip() for x in obj if str(x).strip()]
        elif isinstance(obj, dict):
            # accept { "keywords": [...] } style
            if "keywords" in obj and isinstance(obj["keywords"], list):
                terms = [str(x).strip() for x in obj["keywords"] if str(x).strip()]
    except Exception:
        pass

    if not terms:
        # split by newlines, commas
        parts = []
        for line in raw.replace(",", "\n").split("\n"):
            s = line.strip()
            if s:
                parts.append(s)
        terms = parts

    # de-duplicate while preserving order
    seen = set()
    dedup = []
    for t in terms:
        tt = t.strip()
        if not tt:
            continue
        key = tt.lower()
        if key in seen:
            continue
        seen.add(key)
        dedup.append(tt)

    dedup = dedup[:max_terms]
    if not dedup:
        return ""

    # Quote each term and OR them
    # NOTE: NewsAPI supports boolean operators in q.
    q = " OR ".join([f"\"{t}\"" for t in dedup])
    return f"({q})"

def fetch_newsapi_recent_titles(
    *,
    query: str,
    days_lookback: int = 7,
    language: str = "en",
    page_size: int = 30,
    sort_by: str = "publishedAt",
    domains: str | None = None,      # optional "reuters.com,techcrunch.com"
    exclude_domains: str | None = None,
) -> dict:
    """
    Call NewsAPI and return lightweight evidence:
      {
        "query": str,
        "from": iso,
        "to": iso,
        "articles": [
          {"title","source","url","publishedAt","description"}
        ]
      }
    """
    _ensure_newsapi_key()

    # JST window, but NewsAPI expects ISO; we can pass UTC-ish strings safely.
    now = datetime.now(tz=JST)
    start = now - timedelta(days=days_lookback)

    url = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "from": start.astimezone(timezone.utc).isoformat(),
        "to": now.astimezone(timezone.utc).isoformat(),
        "language": language,
        "pageSize": min(max(int(page_size), 1), 100),
        "sortBy": sort_by,
        "apiKey": NEWSAPI_KEY,
    }
    if domains:
        params["domains"] = domains
    if exclude_domains:
        params["excludeDomains"] = exclude_domains

    r = requests.get(url, params=params, timeout=30)
    # If key is invalid / quota exceeded, surface the payload for debugging.
    try:
        payload = r.json()
    except Exception:
        r.raise_for_status()
        payload = {}

    if r.status_code != 200:
        msg = payload.get("message") if isinstance(payload, dict) else None
        raise RuntimeError(f"NewsAPI error {r.status_code}: {msg or r.text[:200]}")

    articles = payload.get("articles", []) if isinstance(payload, dict) else []
    out = []
    for a in articles:
        if not isinstance(a, dict):
            continue
        src = a.get("source", {}) or {}
        out.append({
            "title": (a.get("title") or "").strip(),
            "source": (src.get("name") or "").strip(),
            "url": (a.get("url") or "").strip(),
            "publishedAt": (a.get("publishedAt") or "").strip(),
            "description": (a.get("description") or "").strip(),
        })

    return {
        "query": query,
        "from": start.isoformat(),
        "to": now.isoformat(),
        "articles": out,
        "total_results": payload.get("totalResults", None),
    }

# --- Lightweight term extraction (English + basic Japanese token capture) ---

_EN_STOPWORDS = {
    "the","a","an","and","or","but","if","then","else","to","of","in","on","for","with","at","by",
    "from","as","is","are","was","were","be","been","being","it","its","this","that","these","those",
    "will","would","can","could","may","might","should","about","over","under","after","before",
    "new","latest","report","reports","says","say","amid","how","why","what","when","where","who",
}

def extract_top_terms_from_titles(
    titles: list[str],
    *,
    top_k: int = 12,
    min_len: int = 3
) -> list[tuple[str, int]]:
    """
    Very simple frequency extraction.
    - English: split on non-letters, lowercase, remove stopwords
    - Japanese: capture contiguous hiragana/katakana/kanji runs length>=2
    Returns: list of (term, count)
    """
    freq: dict[str, int] = {}

    for t in titles:
        if not t:
            continue
        s = str(t)

        # English-ish tokens
        eng_tokens = re.findall(r"[A-Za-z][A-Za-z0-9\-\_]+", s)
        for tok in eng_tokens:
            w = tok.lower().strip("-_")
            if len(w) < min_len:
                continue
            if w in _EN_STOPWORDS:
                continue
            freq[w] = freq.get(w, 0) + 1

        # Japanese-ish tokens (very rough)
        jp_tokens = re.findall(r"[\u3040-\u30ff\u4e00-\u9fff]{2,}", s)
        for tok in jp_tokens:
            w = tok.strip()
            if len(w) < 2:
                continue
            freq[w] = freq.get(w, 0) + 1

    return sorted(freq.items(), key=lambda x: (-x[1], x[0]))[:top_k]

def get_newsapi_signal_for_target(
    *,
    target_id: str,
    search_keywords_text: str,
    days_lookback: int = 7,
    language: str = "en",
    page_size: int = 30
) -> dict:
    """
    Convenience wrapper:
      - build query from Search Keywords
      - fetch recent titles
      - compute top terms
    Returns dict to store in external_signals[target_id]["newsapi"].
    """
    q = build_newsapi_query_from_keywords(search_keywords_text)
    if not q:
        return {
            "query": "",
            "articles": [],
            "titles": [],
            "top_terms": [],
            "note": "empty keywords -> no query",
        }

    data = fetch_newsapi_recent_titles(
        query=q,
        days_lookback=days_lookback,
        language=language,
        page_size=page_size,
        sort_by="publishedAt",
    )

    titles = [a["title"] for a in data["articles"] if a.get("title")]
    top_terms = extract_top_terms_from_titles(titles, top_k=12)

    return {
        "query": data["query"],
        "from": data["from"],
        "to": data["to"],
        "articles": data["articles"],   # lightweight evidence
        "titles": titles,
        "top_terms": top_terms,         # [(term,count), ...]
    }

print("✓ NewsAPI functions loaded (Step 11-3A)")

# ============================================================
# Step 11-3A (Test) — End-to-end on ONE target (NewsAPI only)
# ============================================================
# Overview:
#   Pick one target (prefer from tuning_targets_df), pull its current Search Keywords
#   from targets_df, run NewsAPI fetch, and print a compact diagnostic preview.
#
# Inputs / Outputs:
#   Inputs:  tuning_targets_df, targets_df, get_newsapi_signal_for_target()
#   Outputs: external_signals[target_id]["newsapi"] populated for the chosen target
#
# Notes:
#   - This is a minimal sanity check before scaling to multiple targets.
#

print("Running end-to-end test for ONE target (NewsAPI)...\n")

# --- Pick one target to test ---
def pick_one_target_id() -> str:
    # Prefer tuning candidates (more likely to be interesting)
    if "tuning_targets_df" in globals() and tuning_targets_df is not None and not tuning_targets_df.empty:
        return str(tuning_targets_df.iloc[0]["page_id"])
    # Fallback: just pick first target
    if targets_df is not None and not targets_df.empty:
        return str(targets_df.iloc[0]["page_id"])
    raise RuntimeError("No targets available. Run Cell 05 and Cell 11 Step 11-1 first.")

test_target_id = normalize_uuid(pick_one_target_id())

# --- Pull current config from targets_df ---
row = targets_df[targets_df["page_id"].apply(normalize_uuid) == test_target_id]
if row.empty:
    raise RuntimeError(f"Target not found in targets_df: {test_target_id}")

test_target_name = row.iloc[0].get("name", "Untitled")
test_target_type = row.iloc[0].get("type", "")
test_keywords_text = row.iloc[0].get("search_keywords", "") or ""

print(f"Selected target:")
print(f"  id:   {test_target_id}")
print(f"  name: {test_target_name}")
print(f"  type: {test_target_type}")
print(f"\nCurrent Search Keywords (raw):\n{test_keywords_text}\n")

# --- Run NewsAPI signal fetch ---
try:
    news_sig = get_newsapi_signal_for_target(
        target_id=test_target_id,
        search_keywords_text=test_keywords_text,
        days_lookback=7,
        language="en",
        page_size=30
    )
except Exception as e:
    print(f"⚠ NewsAPI fetch failed: {e}")
    news_sig = {
        "query": "",
        "articles": [],
        "titles": [],
        "top_terms": [],
        "note": f"NewsAPI error: {e}",
    }

# --- Store into external_signals dict ---
if "external_signals" not in globals() or external_signals is None:
    external_signals = {}
external_signals.setdefault(test_target_id, {})
external_signals[test_target_id]["newsapi"] = news_sig

# --- Preview ---
print("NewsAPI result preview:")
print(f"  query: {news_sig.get('query','')}")
print(f"  articles: {len(news_sig.get('articles', []))}")
if news_sig.get("top_terms"):
    print("\n  top_terms (term, count):")
    for term, cnt in news_sig["top_terms"][:10]:
        print(f"    - {term}: {cnt}")
else:
    print("\n  top_terms: (none)")

arts = news_sig.get("articles", []) or []
if arts:
    print("\n  sample articles (up to 5):")
    for a in arts[:5]:
        print(f"    - [{a.get('source','')}] {a.get('title','')}")
        print(f"      {a.get('publishedAt','')} | {a.get('url','')}")
else:
    print("\n  sample articles: (none)")

print("\n✓ One-target end-to-end test completed (NewsAPI only)")

# ============================================================
# Step 11-3B — Google CSE: fetch results + domain distribution
# ============================================================
# Overview:
#   Fetch Google Custom Search results for a query (built from current keywords),
#   then compute domain distribution to detect:
#     - noisy domains dominating results
#     - missing first-party / authoritative sources
#
# Notes:
#   - Uses Google Custom Search JSON API:
#       https://www.googleapis.com/customsearch/v1
#   - Requires:
#       GOOGLE_CSE_CX
#       GOOGLE_API_KEY (or GOOGLE_CSE_API_KEY)
#   - Stores only lightweight evidence: title, link, displayLink, snippet.
#

from urllib.parse import urlparse

GOOGLE_CSE_CX = os.getenv("GOOGLE_CSE_CX")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_CSE_API_KEY")

def _ensure_cse_env() -> None:
    if not GOOGLE_CSE_CX or not str(GOOGLE_CSE_CX).strip():
        raise RuntimeError("GOOGLE_CSE_CX is not set in env.txt (required for Google CSE).")
    if not GOOGLE_API_KEY or not str(GOOGLE_API_KEY).strip():
        raise RuntimeError("GOOGLE_API_KEY (or GOOGLE_CSE_API_KEY) is not set in env.txt (required for Google CSE).")

def _domain_from_url(u: str) -> str:
    try:
        netloc = urlparse(u).netloc.lower()
        if netloc.startswith("www."):
            netloc = netloc[4:]
        return netloc
    except Exception:
        return ""

def build_cse_query_from_keywords(
    keywords_text: str,
    *,
    max_terms: int = 6,
) -> str:
    """
    Build a Google CSE query string from the same 'Search Keywords' field.

    Google CSE supports boolean operators (OR).
    We keep it simple:
      "term1" OR "term2" OR "term3"
    """
    if not keywords_text:
        return ""

    raw = str(keywords_text).strip()

    # Best-effort JSON parse (same behavior as NewsAPI builder)
    terms: list[str] = []
    try:
        obj = json.loads(raw)
        if isinstance(obj, list):
            terms = [str(x).strip() for x in obj if str(x).strip()]
        elif isinstance(obj, dict) and "keywords" in obj and isinstance(obj["keywords"], list):
            terms = [str(x).strip() for x in obj["keywords"] if str(x).strip()]
    except Exception:
        pass

    if not terms:
        parts = []
        for line in raw.replace(",", "\n").split("\n"):
            s = line.strip()
            if s:
                parts.append(s)
        terms = parts

    # de-duplicate while preserving order
    seen = set()
    dedup = []
    for t in terms:
        tt = t.strip()
        if not tt:
            continue
        key = tt.lower()
        if key in seen:
            continue
        seen.add(key)
        dedup.append(tt)

    dedup = dedup[:max_terms]
    if not dedup:
        return ""

    return " OR ".join([f"\"{t}\"" for t in dedup])

def fetch_google_cse_results(
    *,
    query: str,
    num_results: int = 20,
    lr: str | None = None,           # e.g., "lang_en"
    gl: str | None = None,           # e.g., "us"
    safe: str = "off",               # "off" / "active"
) -> dict:
    """
    Fetch Google CSE results (lightweight).
    Returns:
      {
        "query": str,
        "items": [{"title","link","displayLink","snippet"}...],
        "domains": {domain: count, ...}
      }
    """
    _ensure_cse_env()
    url = "https://www.googleapis.com/customsearch/v1"

    # CSE allows up to 10 per request; paginate using start=1,11,21,...
    num_results = max(1, min(int(num_results), 50))
    all_items = []
    start = 1

    while len(all_items) < num_results:
        batch = min(10, num_results - len(all_items))
        params = {
            "key": GOOGLE_API_KEY,
            "cx": GOOGLE_CSE_CX,
            "q": query,
            "num": batch,
            "start": start,
            "safe": safe,
        }
        if lr:
            params["lr"] = lr
        if gl:
            params["gl"] = gl

        r = requests.get(url, params=params, timeout=30)
        payload = r.json() if r.headers.get("content-type","").startswith("application/json") else {}
        if r.status_code != 200:
            msg = payload.get("error", {}).get("message") if isinstance(payload, dict) else None
            raise RuntimeError(f"Google CSE error {r.status_code}: {msg or r.text[:200]}")

        items = payload.get("items", []) if isinstance(payload, dict) else []
        for it in items:
            if not isinstance(it, dict):
                continue
            all_items.append({
                "title": (it.get("title") or "").strip(),
                "link": (it.get("link") or "").strip(),
                "displayLink": (it.get("displayLink") or "").strip(),
                "snippet": (it.get("snippet") or "").strip(),
            })

        if not items:
            break

        start += batch
        if start > 91:  # API limit safety (max start ~ 91 for 10 items)
            break

    # domain distribution
    domains = {}
    for it in all_items:
        d = it.get("displayLink") or _domain_from_url(it.get("link",""))
        if d:
            domains[d] = domains.get(d, 0) + 1

    # sort domains by count desc
    domains_sorted = dict(sorted(domains.items(), key=lambda x: (-x[1], x[0])))

    return {
        "query": query,
        "items": all_items,
        "domains": domains_sorted,
    }

def get_cse_signal_for_target(
    *,
    target_id: str,
    search_keywords_text: str,
    num_results: int = 20,
    lr: str | None = "lang_en",
    gl: str | None = None
) -> dict:
    """
    Convenience wrapper:
      - build CSE query from Search Keywords
      - fetch results
      - compute domain distribution
    Returns dict to store in external_signals[target_id]["cse"].
    """
    q = build_cse_query_from_keywords(search_keywords_text)
    if not q:
        return {
            "query": "",
            "items": [],
            "domains": {},
            "note": "empty keywords -> no query",
        }

    data = fetch_google_cse_results(
        query=q,
        num_results=num_results,
        lr=lr,
        gl=gl,
        safe="off",
    )
    return data

print("✓ Google CSE functions loaded (Step 11-3B)")

# ============================================================
# Step 11-3B (Test) — Extend ONE target test with Google CSE
# ============================================================
print("\nRunning Google CSE test for the same target...\n")

try:
    cse_sig = get_cse_signal_for_target(
        target_id=test_target_id,
        search_keywords_text=test_keywords_text,
        num_results=20,
        lr="lang_en",
        gl=None
    )
except Exception as e:
    print(f"⚠ Google CSE fetch failed: {e}")
    cse_sig = {
        "query": "",
        "items": [],
        "domains": {},
        "note": f"CSE error: {e}",
    }

# Store into external_signals dict
external_signals.setdefault(test_target_id, {})
external_signals[test_target_id]["cse"] = cse_sig

# Preview domains
print("Google CSE result preview:")
print(f"  query: {cse_sig.get('query','')}")
print(f"  items: {len(cse_sig.get('items', []))}")

domains = cse_sig.get("domains", {}) or {}
if domains:
    print("\n  top domains (domain: count):")
    for d, cnt in list(domains.items())[:10]:
        print(f"    - {d}: {cnt}")
else:
    print("\n  top domains: (none)")

items = cse_sig.get("items", []) or []
if items:
    print("\n  sample results (up to 5):")
    for it in items[:5]:
        print(f"    - [{it.get('displayLink','')}] {it.get('title','')}")
        print(f"      {it.get('link','')}")
else:
    print("\n  sample results: (none)")

print("\n✓ One-target end-to-end test completed (NewsAPI + Google CSE)")

# ============================================================
# Step 11-5 — LLM: generate tuning proposals for ONE target
# ============================================================
# Overview:
#   Use LLM as an "editor" to propose concrete changes:
#     - Search Keywords (add/remove/replace)
#     - Source URLs (add/remove)
#   based on:
#     - computed metrics (signal/noise/events)
#     - NewsAPI titles + top terms
#     - Google CSE domain distribution + sample results
#
# Output:
#   llm_tuning = dict (strict JSON)
#   target_tuning_proposals appended (list of proposals)
#
# Notes:
#   - This does NOT write to Notion.
#   - Keep outputs reviewable and diff-style.
#

def build_llm_tuning_prompt_one_target(
    *,
    target_name: str,
    target_type: str,
    target_id: str,
    current_keywords_text: str,
    current_source_urls_text: str,
    metrics_row: Optional[pd.Series],
    news_sig: dict,
    cse_sig: dict
) -> str:
    # metrics
    if metrics_row is not None and not metrics_row.empty:
        m = {
            "number_of_events": int(metrics_row.get("number_of_events", 0) or 0),
            "signal_score": float(metrics_row.get("signal_score", 0.0) or 0.0),
            "noise_score": float(metrics_row.get("noise_score", 0.0) or 0.0),
            "share_action_needed": float(metrics_row.get("share_action_needed", 0.0) or 0.0),
            "dedup_dup_rate": float(metrics_row.get("dedup_dup_rate", 0.0) or 0.0),
            "days_since_last_event": metrics_row.get("days_since_last_event", None),
        }
    else:
        m = None

    # News: take a small sample of titles (avoid huge prompt)
    titles = news_sig.get("titles", []) or []
    titles_sample = titles[:10]
    top_terms = news_sig.get("top_terms", []) or []
    top_terms_sample = top_terms[:12]

    # CSE: domains top
    domains = cse_sig.get("domains", {}) or {}
    domains_top = list(domains.items())[:12]

    # CSE sample results
    items = cse_sig.get("items", []) or []
    items_sample = items[:6]

    # Compose
    return f"""
You are improving a monitoring target configuration for a weekly research OS.

Target:
- Name: {target_name}
- Type: {target_type}
- Target ID: {target_id}

Current configuration (as stored in Notion):
- Search Keywords (raw):
{current_keywords_text if current_keywords_text else "(empty)"}

- Source URLs (raw):
{current_source_urls_text if current_source_urls_text else "(empty / not set)"}

Computed monitoring performance (past 7 days):
{json.dumps(m, ensure_ascii=False) if m is not None else "(metrics unavailable)"}

External signals from NewsAPI (last 7 days):
- Query used: {news_sig.get("query","")}
- Sample titles (up to 10):
{json.dumps(titles_sample, ensure_ascii=False)}

- Top terms from titles (term,count):
{json.dumps(top_terms_sample, ensure_ascii=False)}

External signals from Google CSE:
- Query used: {cse_sig.get("query","")}
- Top domains (domain,count):
{json.dumps(domains_top, ensure_ascii=False)}

- Sample results (title,displayLink,link):
{json.dumps(items_sample, ensure_ascii=False)}

Diagnosis hints (what we already observe):
- NewsAPI results look noisy because the keyword "grant" is too generic (e.g., Cary Grant / sports).
- CSE results are mostly relevant (NIH/NSF), but there is at least one potentially wrong domain ("nsf.org" is not NSF.gov).

Task:
Propose concrete tuning changes to improve precision and reduce noise while keeping coverage.
You may propose changes for:
1) Search Keywords
2) Source URLs (or domain preferences)

Rules:
- Be specific and operational (add/remove/replace keywords; add/remove URLs).
- Prefer authoritative sources for POLICY targets (agency domains, official grants pages, federal register, etc.).
- Keep proposed keywords short and boolean-friendly.
- Avoid generic terms that create entertainment/sports noise.
- Return STRICT JSON ONLY.

Output JSON schema:
{{
  "keyword_update": {{
    "add": ["..."],
    "remove": ["..."],
    "replace": [{{"from": "...", "to": "..."}}]
  }},
  "source_url_update": {{
    "add": ["..."],
    "remove": ["..."],
    "notes": "optional short note about domain preferences"
  }},
  "change_summary": {{
    "keywords": "diff-style short summary",
    "source_urls": "diff-style short summary"
  }},
  "rationale": "2-4 sentences explaining why these changes reduce noise and improve coverage"
}}
""".strip()

def run_llm_tuning_one_target(
    *,
    prompt: str,
    model: str,
    temperature: float,
    max_tokens: int = 500
) -> dict:
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a precise editor. Output STRICT JSON only."},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    text = (resp.choices[0].message.content or "").strip()

    # robust JSON extraction (reuse helper if you already defined one)
    obj = None
    try:
        obj = json.loads(text)
    except Exception:
        # try first {...}
        s = text
        a = s.find("{")
        b = s.rfind("}")
        if a >= 0 and b > a:
            obj = json.loads(s[a:b+1])

    if not isinstance(obj, dict):
        raise ValueError(f"LLM returned non-JSON: {text[:200]}")
    return obj

# --- Prepare inputs for this target ---
test_target_id = normalize_uuid(test_target_id)

# metrics row for this target (if available)
mrow_df = metrics_df[metrics_df["page_id"].apply(normalize_uuid) == test_target_id]
metrics_row = mrow_df.iloc[0] if not mrow_df.empty else None

# current source urls:
# NOTE: In your current targets_df from Cell 05, you may NOT have "source_urls" loaded yet.
# If not present, keep blank. Later we’ll extend Cell 05 extractor to include it.
current_source_urls_text = ""
if "source_urls" in targets_df.columns:
    current_source_urls_text = str(row.iloc[0].get("source_urls", "") or "")
elif "Source URLs" in targets_df.columns:
    current_source_urls_text = str(row.iloc[0].get("Source URLs", "") or "")

# fetch news/cse signals from external_signals (from previous test)
news_sig = external_signals.get(test_target_id, {}).get("newsapi", {}) if "external_signals" in globals() else {}
cse_sig = external_signals.get(test_target_id, {}).get("cse", {}) if "external_signals" in globals() else {}

prompt = build_llm_tuning_prompt_one_target(
    target_name=test_target_name,
    target_type=test_target_type,
    target_id=test_target_id,
    current_keywords_text=test_keywords_text,
    current_source_urls_text=current_source_urls_text,
    metrics_row=metrics_row,
    news_sig=news_sig,
    cse_sig=cse_sig
)

print("Calling LLM for tuning proposal...\n")
llm_tuning = run_llm_tuning_one_target(
    prompt=prompt,
    model=LLM_MODEL,
    temperature=LLM_TEMPERATURE,
    max_tokens=600
)

print("✓ LLM tuning proposal received\n")
print(json.dumps(llm_tuning, ensure_ascii=False, indent=2))

# --- Convert into proposal objects (notion write comes in Step 11-6 / Step 12) ---
def _as_lines(x) -> str:
    if not x:
        return ""
    if isinstance(x, list):
        return "\n".join([str(v) for v in x if str(v).strip()])
    return str(x)

keyword_update = llm_tuning.get("keyword_update", {}) or {}
src_update = llm_tuning.get("source_url_update", {}) or {}
summary = llm_tuning.get("change_summary", {}) or {}
rationale = llm_tuning.get("rationale", "") or ""

# Proposed keywords: apply add/remove/replace to a list (best-effort)
# Current keywords -> list
cur_kw_parts = [p.strip() for p in str(test_keywords_text).replace(",", "\n").split("\n") if p.strip()]
cur_kw = []
seen = set()
for k in cur_kw_parts:
    lk = k.lower()
    if lk in seen:
        continue
    seen.add(lk)
    cur_kw.append(k)

# Apply replacements
for rep in (keyword_update.get("replace", []) or []):
    if isinstance(rep, dict) and rep.get("from") and rep.get("to"):
        frm = str(rep["from"]).strip()
        to = str(rep["to"]).strip()
        cur_kw = [to if x.strip().lower() == frm.lower() else x for x in cur_kw]

# Remove
rem = set([str(x).strip().lower() for x in (keyword_update.get("remove", []) or []) if str(x).strip()])
cur_kw = [x for x in cur_kw if x.strip().lower() not in rem]

# Add
adds = []
for x in (keyword_update.get("add", []) or []):
    xx = str(x).strip()
    if not xx:
        continue
    if xx.lower() in [y.lower() for y in cur_kw]:
        continue
    adds.append(xx)

proposed_kw = cur_kw + adds

# Proposed source urls: we cannot apply cleanly if current is empty; just store add/remove lists.
proposed_src_text = ""
if src_update:
    proposed_src_text = (
        "ADD:\n" + _as_lines(src_update.get("add", [])) +
        ("\n\nREMOVE:\n" + _as_lines(src_update.get("remove", [])) if src_update.get("remove") else "") +
        ("\n\nNOTES:\n" + str(src_update.get("notes","")).strip() if str(src_update.get("notes","")).strip() else "")
    ).strip()

# Add proposals (aligned with your weekly_target_update new fields)
if "target_tuning_proposals" not in globals() or target_tuning_proposals is None:
    target_tuning_proposals = []

# 1) Keyword proposal
target_tuning_proposals.append({
    "target_id": test_target_id,
    "target_name": test_target_name,
    "proposal_type": "Keyword Update",
    "field": "Search Keywords",
    "current_value": str(test_keywords_text),
    "proposed_value": "\n".join(proposed_kw),
    "change_summary": str(summary.get("keywords","")).strip(),
    "rationale": rationale,
    "evidence": {
        "newsapi_query": news_sig.get("query",""),
        "newsapi_top_terms": news_sig.get("top_terms", [])[:12],
        "cse_top_domains": list((cse_sig.get("domains", {}) or {}).items())[:10],
    }
})

# 2) Source URL proposal (only if non-empty)
if proposed_src_text:
    target_tuning_proposals.append({
        "target_id": test_target_id,
        "target_name": test_target_name,
        "proposal_type": "Source Update",
        "field": "Source URLs",
        "current_value": str(current_source_urls_text),
        "proposed_value": proposed_src_text,
        "change_summary": str(summary.get("source_urls","")).strip(),
        "rationale": rationale,
        "evidence": {
            "cse_top_domains": list((cse_sig.get("domains", {}) or {}).items())[:10],
        }
    })

print("\n✓ Proposals appended to target_tuning_proposals")
print(f"  total proposals so far: {len(target_tuning_proposals)}")

# ============================================================
# Step 11-5 (Refine) — Prompt refinement + re-run for ONE target
# ============================================================

def build_llm_tuning_prompt_one_target_refined(
    *,
    target_name: str,
    target_type: str,
    target_id: str,
    current_keywords_text: str,
    current_source_urls_text: str,
    metrics_row: Optional[pd.Series],
    news_sig: dict,
    cse_sig: dict
) -> str:
    # metrics (same as before)
    if metrics_row is not None and not metrics_row.empty:
        m = {
            "number_of_events": int(metrics_row.get("number_of_events", 0) or 0),
            "signal_score": float(metrics_row.get("signal_score", 0.0) or 0.0),
            "noise_score": float(metrics_row.get("noise_score", 0.0) or 0.0),
            "share_action_needed": float(metrics_row.get("share_action_needed", 0.0) or 0.0),
            "dedup_dup_rate": float(metrics_row.get("dedup_dup_rate", 0.0) or 0.0),
            "days_since_last_event": metrics_row.get("days_since_last_event", None),
        }
    else:
        m = None

    titles = (news_sig.get("titles", []) or [])[:12]
    top_terms = (news_sig.get("top_terms", []) or [])[:15]

    domains = cse_sig.get("domains", {}) or {}
    domains_top = list(domains.items())[:15]

    items = (cse_sig.get("items", []) or [])[:8]

    return f"""
You are improving a monitoring target configuration for a weekly research OS.

Target:
- Name: {target_name}
- Type: {target_type}
- Target ID: {target_id}

Current configuration (Notion):
- Search Keywords (raw):
{current_keywords_text if current_keywords_text else "(empty)"}

- Source URLs (raw):
{current_source_urls_text if current_source_urls_text else "(empty / not set)"}

Computed monitoring performance (past 7 days):
{json.dumps(m, ensure_ascii=False) if m is not None else "(metrics unavailable)"}

External signals:
NewsAPI (last 7 days):
- Query: {news_sig.get("query","")}
- Sample titles:
{json.dumps(titles, ensure_ascii=False)}
- Top terms (term,count):
{json.dumps(top_terms, ensure_ascii=False)}

Google CSE:
- Query: {cse_sig.get("query","")}
- Top domains (domain,count):
{json.dumps(domains_top, ensure_ascii=False)}
- Sample results (title,displayLink,link):
{json.dumps(items, ensure_ascii=False)}

Hard constraints (must follow):

0) Source evaluation rules (VERY IMPORTANT):
   - Do NOT assume government (.gov) domains are always preferred.
   - Preferred source types depend on the Target Type:

     • POLICY targets:
       - Prefer official government or regulatory sources when relevant
         (e.g., agency pages, official announcements, Federal Register, grants portals).

     • VC targets:
       - Prefer the firm's official website, blog/newsroom, press releases,
         portfolio announcements, and founder/investment memos.
       - Government domains are usually NOT relevant unless directly related.

     • STARTUP targets:
       - Prefer official company blog/newsroom, product updates, documentation,
         GitHub, changelogs, and investor announcements.
       - Avoid unrelated government sources.

     • PEOPLE targets:
       - Prefer personal blogs, Substack/Medium posts, interviews/podcasts,
         academic pages, and verified social accounts.

   - When proposing Source URL updates:
     - Prioritize relevance and signal quality over domain type.
     - Do NOT replace a relevant official site with an unrelated “authoritative” domain.
     - If the current source URLs are already relevant to the target's role,
       do NOT propose replacing them unless there is clear evidence of noise or irrelevance.

1) Keywords:
   - Remove overly generic terms that cause entertainment/sports/person-name noise (e.g., "grant").
   - Keep the list compact (<= 10 total terms after changes).
   - Keyword specificity should match Target Type:
     • POLICY: grants/regulatory vocabulary is OK (NOFO, FOA, RFA, solicitation, etc.).
     • VC/STARTUP/PEOPLE: avoid policy-only jargon unless the target is explicitly policy-adjacent.

2) Source URLs:
   - Prefer authoritative sources *appropriate to the Target Type* (see rule 0).
   - If Google CSE shows confusing look-alike domains (e.g., nsf.org vs nsf.gov),
     explicitly REMOVE the wrong one(s) *when it is a look-alike / irrelevant*.
   - Do NOT remove authoritative RSS feeds unless clearly irrelevant; RSS can be a primary source.
   - Ensure URLs are valid-looking and correctly formatted (no wrong hostnames like "www.grants.nih.gov" if not real).

3) Output format:
   - Return STRICT JSON ONLY (no markdown).
   - change_summary MUST be diff-style using +add/-remove/replace.

Task:
Propose concrete tuning changes for:
- Search Keywords
- Source URLs

Output JSON schema:
{{
  "keyword_update": {{
    "add": ["..."],
    "remove": ["..."],
    "replace": [{{"from":"...","to":"..."}}]
  }},
  "source_url_update": {{
    "add": ["..."],
    "remove": ["..."],
    "notes": "short note about domain preferences"
  }},
  "change_summary": {{
    "keywords": "e.g., +add: X; -remove: Y; replace: A->B",
    "source_urls": "e.g., +add: X; -remove: Y"
  }},
  "rationale": "2-4 sentences"
}}
""".strip()


prompt2 = build_llm_tuning_prompt_one_target_refined(
    target_name=test_target_name,
    target_type=test_target_type,
    target_id=test_target_id,
    current_keywords_text=test_keywords_text,
    current_source_urls_text=current_source_urls_text,
    metrics_row=metrics_row,
    news_sig=news_sig,
    cse_sig=cse_sig
)

print("Re-calling LLM with refined constraints...\n")
llm_tuning_refined = run_llm_tuning_one_target(
    prompt=prompt2,
    model=LLM_MODEL,
    temperature=0.0,
    max_tokens=600
)

print("✓ Refined LLM tuning proposal received\n")
print(json.dumps(llm_tuning_refined, ensure_ascii=False, indent=2))

# ============================================================
# Step 11-4 — Diagnose monitoring issues per target
# ============================================================
# Based on:
#   - metrics_df (signal_score, noise_ratio, event_count)
#   - external_signals (titles, domains)
#
# Infer likely issues:
#   - Keywords too broad / too generic
#   - Keywords outdated (topic drift)
#   - Missing sub-topics or actors
#   - Source URLs too noisy or too narrow
#
# Output:
#   tuning_diagnostics[target_id] = {
#       "issues": [...],
#       "observations": [...]
#   }
#

tuning_diagnostics = {}

# ============================================================
# Step 11-5 — Generate tuning proposals (LLM-assisted)
# ============================================================
# Translate diagnostics into concrete, reviewable proposals:
#
# For each proposal:
#   - Field: "Search Keywords" or "Source URLs"
#   - Current Value
#   - Proposed Value
#   - Change Summary (diff-style)
#   - Rationale (short explanation)
#
# IMPORTANT:
#   - LLM is used as a *rewriter / editor*, not a decision-maker.
#   - Proposals must be explicit and actionable.
#

# TODO:
#   - Build per-target prompt with:
#       * current config
#       * diagnostics
#       * external signal samples
#   - Parse structured JSON output
#
# Example output item:
# {
#   "target_id": "...",
#   "field": "Search Keywords",
#   "current_value": "...",
#   "proposed_value": "...",
#   "change_summary": "+ add: ... / - remove: ...",
#   "rationale": "...",
#   "evidence": "NewsAPI / Google CSE"
# }

target_tuning_proposals = []

# ============================================================
# Step 11-6 — Normalize proposals for weekly_target_update DB
# ============================================================
# Convert proposals into rows aligned with weekly_target_update schema:
#
# Required fields:
#   - Name (auto-generated)
#   - Target (relation)
#   - Proposal Type (e.g., "Keyword Update", "Source Update")
#   - Field
#   - Current Value
#   - Proposed Value
#   - Change Summary
#   - Rationale
#   - Confidence
#   - Evidence Events (optional)
#   - Week (relation)
#   - Status = "Proposed"
#

weekly_target_update_rows = []

# ============================================================
# Step 11-7 — Preview proposals (NO WRITE)
# ============================================================
# Display proposals for human review before any Notion write.
#

print("\n✓ Target tuning skeleton ready")
print("  - No external APIs called yet")
print("  - No LLM calls yet")
print("  - No Notion writes performed")


Generating target tuning suggestions (Keywords / Source URLs)...

Identified 23 targets for potential tuning
✓ NewsAPI functions loaded (Step 11-3A)
Running end-to-end test for ONE target (NewsAPI)...

Selected target:
  id:   2f78e0e4-d162-81a9-b8e6-eebf6cea6def
  name: GRANTSGOV
  type: POLICY

Current Search Keywords (raw):
grant, NSF, NIH

NewsAPI result preview:
  query: ("grant" OR "NSF" OR "NIH")
  articles: 29

  top_terms (term, count):
    - best: 8
    - supplements: 7
    - claims: 2
    - democratic: 2
    - feds: 2
    - fraud: 2
    - funds: 2
    - grant: 2
    - judge: 2
    - nih: 2

  sample articles (up to 5):
    - [CinemaBlend] ‘Might As Well Be At An Amusement Park.’ See What Critics Are Saying About Keke Palmer’s The ‘Burbs
      2026-02-07T02:13:38Z | https://www.cinemablend.com/streaming-news/might-as-well-be-at-an-amusement-park-see-what-critics-are-saying-about-keke-palmers-the-burbs
    - [Legalinsurrection.com] Appeals Court Greenlights Trump Anti-DEI Exec

In [38]:
# ============================================================
# Cell 12 — Write tuning proposals to weekly_target_update (Notion)
# ============================================================
# Overview:
#   Create two proposal pages in weekly_target_update DB:
#     1) Field=Search Keywords (Keyword Update)
#     2) Field=Source URLs (Source Update)
#
# Inputs / Outputs:
#   Inputs:
#     - llm_tuning_refined (from Step 11-5 refine)
#     - test_target_id, test_target_name
#     - weekly_target_update data_source_id resolved in Cell 03
#     - WEEK_PAGE_ID (relation target)
#   Outputs:
#     - created_pages: list of created Notion page objects (lightweight)
#
# Notes:
#   - Uses POST /v1/pages to create pages.
#   - Does NOT modify the monitoring target itself.
#
# ============================================================
# Resolve current Week page from Weekly Digests DB
# ============================================================

print("Resolving current Week page from Weekly Digests DB...\n")

WEEKLY_DIGESTS_DB_ID = normalize_uuid(
    os.getenv("NOTION_WEEKLY_DIGESTS_DB_ID", "")
)

if not WEEKLY_DIGESTS_DB_ID:
    raise RuntimeError("NOTION_WEEKLY_DIGESTS_DB_ID is not set in env.txt")

# ① data_source_id は Cell 03 で解決済みと仮定
weekly_digests_ds = RESOLVED_DB["weekly_digests"]["data_source_id"]

# ② 最新1件を取得（並び順は Notion 側のデフォルト＝更新順でOK）
resp = notion_post(
    f"/v1/data_sources/{weekly_digests_ds}/query",
    {
        "page_size": 1
    }
)

results = resp.get("results", [])
if not results:
    raise RuntimeError("Weekly Digests DB has no pages")

WEEK_PAGE_ID = normalize_uuid(results[0]["id"])

print(f"✓ Current Week page resolved:")
print(f"  page_id: {WEEK_PAGE_ID}")
print(f"  title:   {results[0]['properties'].get('Name', {}).get('title', [{}])[0].get('plain_text', '')}")


# WEEK_PAGE_ID is already resolved by the previous cell

# --- Resolve weekly_target_update database_id (not data_source_id) for page creation ---
# For POST /v1/pages, we need parent.database_id (the DB id), not data_source_id.
WEEKLY_TARGET_UPDATE_DB_ID = normalize_uuid(NOTION_WEEKLY_TARGET_UPDATE_DB_ID)

if not WEEKLY_TARGET_UPDATE_DB_ID:
    raise RuntimeError("NOTION_WEEKLY_TARGET_UPDATE_DB_ID is missing. Check env.txt.")

# --- Helpers to build Notion property payloads ---

def rt(text: str) -> list:
    """Notion rich_text array."""
    text = "" if text is None else str(text)
    if not text.strip():
        return []
    return [{"type": "text", "text": {"content": text}}]

def title(text: str) -> list:
    """Notion title array."""
    text = "" if text is None else str(text)
    if not text.strip():
        text = "Untitled"
    return [{"type": "text", "text": {"content": text}}]

def select(name: str) -> dict:
    """Notion select payload."""
    if not name:
        return None
    return {"name": name}

def relation(ids: list[str]) -> list[dict]:
    """Notion relation array."""
    out = []
    for i in ids:
        if i and str(i).strip():
            out.append({"id": normalize_uuid(str(i))})
    return out

def create_weekly_target_update_page(
    *,
    name: str,
    target_page_id: str,
    week_page_id: str,
    proposal_type: str,
    field: str,
    current_value: str,
    proposed_value: str,
    change_summary: str,
    rationale: str,
    confidence: float = 0.8,
    status: str = "Proposed",
    evidence_event_ids: list[str] | None = None,
) -> dict:
    """
    Create one proposal page in weekly_target_update DB.
    """
    props = {
        "Name": {"title": title(name)},
        "Target": {"relation": relation([target_page_id])},
        "Week": {"relation": relation([week_page_id])},
        "Proposal Type": {"select": select(proposal_type)},
        "Field": {"select": select(field)},
        "Current Value": {"rich_text": rt(current_value)},
        "Proposed Value": {"rich_text": rt(proposed_value)},
        "Change Summary": {"rich_text": rt(change_summary)},
        "Rationale": {"rich_text": rt(rationale)},
        "Confidence": {"number": float(confidence)},
        "Status": {"select": select(status)},
    }

    if evidence_event_ids:
        props["Evidence Events"] = {"relation": relation(evidence_event_ids)}

    payload = {
        "parent": {"database_id": WEEKLY_TARGET_UPDATE_DB_ID},
        "properties": props
    }

    return notion_post("/v1/pages", payload)

# --- Build the two proposals from llm_tuning_refined ---

# 1) Search Keywords proposal
kw_u = llm_tuning_refined.get("keyword_update", {}) or {}
kw_summary = (llm_tuning_refined.get("change_summary", {}) or {}).get("keywords", "")
rationale = llm_tuning_refined.get("rationale", "") or ""

# Reconstruct proposed keywords as newline-separated list:
# current keywords raw:
current_keywords_text = test_keywords_text

# apply updates (best-effort)
cur = [p.strip() for p in str(current_keywords_text).replace(",", "\n").split("\n") if p.strip()]

# replace
for rep in (kw_u.get("replace", []) or []):
    if isinstance(rep, dict) and rep.get("from") and rep.get("to"):
        frm = str(rep["from"]).strip().lower()
        to = str(rep["to"]).strip()
        cur = [to if x.strip().lower() == frm else x for x in cur]

# remove
rem = {str(x).strip().lower() for x in (kw_u.get("remove", []) or []) if str(x).strip()}
cur = [x for x in cur if x.strip().lower() not in rem]

# add
adds = []
for x in (kw_u.get("add", []) or []):
    xx = str(x).strip()
    if not xx:
        continue
    if xx.lower() in [y.lower() for y in cur]:
        continue
    adds.append(xx)

proposed_keywords_text = "\n".join(cur + adds)

# 2) Source URLs proposal
src_u = llm_tuning_refined.get("source_url_update", {}) or {}
src_summary = (llm_tuning_refined.get("change_summary", {}) or {}).get("source_urls", "")

# Current source urls: try to pull if you have it; otherwise keep blank
current_source_urls_text = ""
if "source_urls" in targets_df.columns:
    current_source_urls_text = str(targets_df.loc[targets_df["page_id"].apply(normalize_uuid) == test_target_id].iloc[0].get("source_urls", "") or "")

proposed_source_urls_text = "\n".join([u for u in (src_u.get("add", []) or []) if str(u).strip()])
# Keep removals as summary rather than deleting: in proposals we store before/after + summary.
# If you want a fully materialized after-list, you'll need to parse current urls list too.

# --- Create pages ---
created_pages = []

print("Creating weekly_target_update proposal pages...\n")

# Name convention
base = f"{test_target_name} — {datetime.now(tz=JST).strftime('%Y-%m-%d')}"

# Keyword Update page
created_pages.append(
    create_weekly_target_update_page(
        name=f"{base} — Keyword Update",
        target_page_id=test_target_id,
        week_page_id=WEEK_PAGE_ID,
        proposal_type="Keyword Update",
        field="Search Keywords",
        current_value=current_keywords_text,
        proposed_value=proposed_keywords_text,
        change_summary=kw_summary,
        rationale=rationale,
        confidence=0.85,
        status="Proposed",
    )
)

# Source Update page
created_pages.append(
    create_weekly_target_update_page(
        name=f"{base} — Source Update",
        target_page_id=test_target_id,
        week_page_id=WEEK_PAGE_ID,
        proposal_type="Source Update",
        field="Source URLs",
        current_value=current_source_urls_text,
        proposed_value=proposed_source_urls_text,
        change_summary=src_summary,
        rationale=rationale,
        confidence=0.85,
        status="Proposed",
    )
)

print(f"✓ Created {len(created_pages)} proposal pages in weekly_target_update")
for p in created_pages:
    pid = p.get("id", "")
    url = p.get("url", "")
    print(f"  - {pid} | {url}")


Resolving current Week page from Weekly Digests DB...

✓ Current Week page resolved:
  page_id: 2ff8e0e4-d162-8165-9a05-e4061df44fb5
  title:   Weekly Events Digest: 2026-02-02–2026-02-08
Creating weekly_target_update proposal pages...

✓ Created 2 proposal pages in weekly_target_update
  - 3018e0e4-d162-81c3-9676-d84c4f93f887 | https://www.notion.so/GRANTSGOV-2026-02-08-Keyword-Update-3018e0e4d16281c39676d84c4f93f887
  - 3018e0e4-d162-81e6-95a8-f65df8621f54 | https://www.notion.so/GRANTSGOV-2026-02-08-Source-Update-3018e0e4d16281e695a8f65df8621f54


In [39]:
# ============================================================
# Cell 13 — Batch: generate + write tuning proposals for all tuning targets
# ============================================================
# Overview:
#   For each target in tuning_targets_df:
#     - fetch NewsAPI signals
#     - fetch Google CSE signals
#     - call LLM (refined prompt)
#     - create weekly_target_update pages (Keyword Update + Source Update)
#   with de-dup protection (Week x Target x Field).
#
# Notes:
#   - Uses POST /v1/data_sources/{weekly_target_update_ds}/query to check duplicates.
#   - Uses POST /v1/pages to create pages.
#   - Adds basic sleep to avoid API bursts.
#

import time

weekly_target_update_ds = RESOLVED_DB["weekly_target_update"]["data_source_id"]

# --- Helper: build "equals relation contains page id" filter for data_sources query ---
def _relation_contains(property_name: str, page_id: str) -> dict:
    return {
        "property": property_name,
        "relation": {"contains": normalize_uuid(page_id)}
    }

def _select_equals(property_name: str, value: str) -> dict:
    return {
        "property": property_name,
        "select": {"equals": value}
    }

def _and(*conds) -> dict:
    return {"and": [c for c in conds if c]}

def weekly_update_exists(*, week_page_id: str, target_page_id: str, field_value: str) -> bool:
    """
    Check whether a weekly_target_update page already exists for:
      Week contains week_page_id AND Target contains target_page_id AND Field == field_value
    """
    payload = {
        "page_size": 1,
        "filter": _and(
            _relation_contains("Week", week_page_id),
            _relation_contains("Target", target_page_id),
            _select_equals("Field", field_value),
        )
    }
    resp = notion_post(f"/v1/data_sources/{weekly_target_update_ds}/query", payload)
    return bool(resp.get("results"))

def safe_get_target_row(target_id: str) -> pd.Series:
    r = targets_df[targets_df["page_id"].apply(normalize_uuid) == normalize_uuid(target_id)]
    if r.empty:
        raise RuntimeError(f"Target not found in targets_df: {target_id}")
    return r.iloc[0]

def _get_metrics_row(target_id: str) -> Optional[pd.Series]:
    if "metrics_df" not in globals() or metrics_df is None or metrics_df.empty:
        return None
    r = metrics_df[metrics_df["page_id"].apply(normalize_uuid) == normalize_uuid(target_id)]
    return r.iloc[0] if not r.empty else None

def _current_keywords_text_from_row(r: pd.Series) -> str:
    # You may need to adjust the column name depending on how Cell 05 parsed it.
    for col in ["search_keywords", "Search Keywords"]:
        if col in r.index:
            return str(r.get(col, "") or "")
    return ""

def _current_source_urls_text_from_row(r: pd.Series) -> str:
    for col in ["source_urls", "Source URLs"]:
        if col in r.index:
            return str(r.get(col, "") or "")
    return ""

# --- Batch loop ---
print("Batch tuning: generating + writing weekly_target_update proposals...\n")

created = 0
skipped = 0
errors = 0

# You can cap batch size during testing
MAX_TARGETS = None  # e.g., 5 for test
rows = tuning_targets_df if MAX_TARGETS is None else tuning_targets_df.head(MAX_TARGETS)

for i, trow in rows.reset_index(drop=True).iterrows():
    try:
        target_id = normalize_uuid(str(trow["page_id"]))
        tr = safe_get_target_row(target_id)

        target_name = str(tr.get("name", "Untitled"))
        target_type = str(tr.get("type", ""))

        cur_kw_text = _current_keywords_text_from_row(tr)
        cur_src_text = _current_source_urls_text_from_row(tr)

        print(f"\n[{i+1}/{len(rows)}] {target_name} ({target_type})")

        # --- de-dup check (skip if both already exist) ---
        kw_exists = weekly_update_exists(
            week_page_id=WEEK_PAGE_ID,
            target_page_id=target_id,
            field_value="Search Keywords"
        )
        src_exists = weekly_update_exists(
            week_page_id=WEEK_PAGE_ID,
            target_page_id=target_id,
            field_value="Source URLs"
        )

        if kw_exists and src_exists:
            print("  - skip: both proposals already exist for this week")
            skipped += 1
            continue

        # --- fetch signals ---
        news_sig = get_newsapi_signal_for_target(
            target_id=target_id,
            search_keywords_text=cur_kw_text,
            days_lookback=7,
            language="en",
            page_size=30
        )

        cse_sig = get_cse_signal_for_target(
            target_id=target_id,
            search_keywords_text=cur_kw_text,
            num_results=20,
            lr="lang_en",
            gl=None
        )

        # --- call refined prompt ---
        mrow = _get_metrics_row(target_id)

        prompt = build_llm_tuning_prompt_one_target_refined(
            target_name=target_name,
            target_type=target_type,
            target_id=target_id,
            current_keywords_text=cur_kw_text,
            current_source_urls_text=cur_src_text,
            metrics_row=mrow,
            news_sig=news_sig,
            cse_sig=cse_sig
        )

        llm_out = run_llm_tuning_one_target(
            prompt=prompt,
            model=LLM_MODEL,
            temperature=0.0,
            max_tokens=650
        )

        # --- materialize proposed keywords ---
        kw_u = llm_out.get("keyword_update", {}) or {}
        cur = [p.strip() for p in str(cur_kw_text).replace(",", "\n").split("\n") if p.strip()]

        for rep in (kw_u.get("replace", []) or []):
            if isinstance(rep, dict) and rep.get("from") and rep.get("to"):
                frm = str(rep["from"]).strip().lower()
                to = str(rep["to"]).strip()
                cur = [to if x.strip().lower() == frm else x for x in cur]

        rem = {str(x).strip().lower() for x in (kw_u.get("remove", []) or []) if str(x).strip()}
        cur = [x for x in cur if x.strip().lower() not in rem]

        adds = []
        for x in (kw_u.get("add", []) or []):
            xx = str(x).strip()
            if not xx:
                continue
            if xx.lower() in [y.lower() for y in cur]:
                continue
            adds.append(xx)

        proposed_kw_text = "\n".join(cur + adds)

        # --- proposed urls text ---
        src_u = llm_out.get("source_url_update", {}) or {}
        proposed_src_text = "\n".join([str(u).strip() for u in (src_u.get("add", []) or []) if str(u).strip()])

        summaries = llm_out.get("change_summary", {}) or {}
        rationale = llm_out.get("rationale", "") or ""

        base = f"{target_name} — {datetime.now(tz=JST).strftime('%Y-%m-%d')}"

        # --- write pages (respect de-dup per field) ---
        if not kw_exists:
            _ = create_weekly_target_update_page(
                name=f"{base} — Keyword Update",
                target_page_id=target_id,
                week_page_id=WEEK_PAGE_ID,
                proposal_type="Keyword Update",
                field="Search Keywords",
                current_value=cur_kw_text,
                proposed_value=proposed_kw_text,
                change_summary=str(summaries.get("keywords","")).strip(),
                rationale=rationale,
                confidence=0.8,
                status="Proposed",
            )
            print("  - wrote: Keyword Update")
            created += 1
        else:
            print("  - skip: Keyword Update exists")

        if not src_exists:
            _ = create_weekly_target_update_page(
                name=f"{base} — Source Update",
                target_page_id=target_id,
                week_page_id=WEEK_PAGE_ID,
                proposal_type="Source Update",
                field="Source URLs",
                current_value=cur_src_text,
                proposed_value=proposed_src_text,
                change_summary=str(summaries.get("source_urls","")).strip(),
                rationale=rationale,
                confidence=0.8,
                status="Proposed",
            )
            print("  - wrote: Source Update")
            created += 1
        else:
            print("  - skip: Source Update exists")

        # polite rate limiting
        time.sleep(0.8)

    except Exception as e:
        errors += 1
        print(f"  ⚠ error: {e}")
        # continue to next target
        time.sleep(0.5)
        continue

print("\n✓ Batch tuning completed")
print(f"  created pages: {created}")
print(f"  skipped targets: {skipped}")
print(f"  errors: {errors}")


Batch tuning: generating + writing weekly_target_update proposals...


[1/23] GRANTSGOV (POLICY)
  - skip: both proposals already exist for this week

[2/23] 内閣府 科学技術政策 (POLICY)
  - wrote: Keyword Update
  - wrote: Source Update

[3/23] Japan Science and Technology Agency (POLICY)
  - wrote: Keyword Update
  - wrote: Source Update

[4/23] NEDO (POLICY)
  - wrote: Keyword Update
  - wrote: Source Update

[5/23] AMED (POLICY)
  - wrote: Keyword Update
  - wrote: Source Update

[6/23] perplexity (STARTUP)
  - wrote: Keyword Update
  - wrote: Source Update

[7/23] Reid Hoffman (PEOPLE)
  - wrote: Keyword Update
  - wrote: Source Update

[8/23] Masayoshi Son (PEOPLE)
  - wrote: Keyword Update
  - wrote: Source Update

[9/23] Peter Thiel (PEOPLE)
  - wrote: Keyword Update
  - wrote: Source Update

[10/23] Eduardo Saverin (PEOPLE)
  - wrote: Keyword Update
  - wrote: Source Update

[11/23] UTEC (VC)
  - wrote: Keyword Update
  - wrote: Source Update

[12/23] Global Brain (VC)
  - wrote: Keywo

In [45]:
# ============================================================
# Cell 13.5 — Build proposals_df from weekly_target_update DB (this week)
# ============================================================

weekly_target_update_ds = RESOLVED_DB["weekly_target_update"]["data_source_id"]

def query_weekly_target_updates_for_week(week_page_id: str, page_size: int = 100) -> list[dict]:
    all_rows = []
    start_cursor = None
    has_more = True

    while has_more:
        payload = {
            "page_size": page_size,
            "filter": {
                "property": "Week",
                "relation": {"contains": normalize_uuid(week_page_id)}
            }
        }
        if start_cursor:
            payload["start_cursor"] = start_cursor

        resp = notion_post(f"/v1/data_sources/{weekly_target_update_ds}/query", payload)
        results = resp.get("results", [])
        all_rows.extend(results)

        has_more = bool(resp.get("has_more"))
        start_cursor = resp.get("next_cursor")

    return all_rows

def _get_plain_text_title(prop: dict) -> str:
    arr = (prop or {}).get("title", [])
    return arr[0].get("plain_text", "") if arr else ""

def _get_plain_text_rich(prop: dict) -> str:
    arr = (prop or {}).get("rich_text", [])
    return arr[0].get("plain_text", "") if arr else ""

def _get_select_name(prop: dict) -> str:
    s = (prop or {}).get("select")
    return (s or {}).get("name", "") if s else ""

def _get_number(prop: dict):
    return (prop or {}).get("number", None)

pages = query_weekly_target_updates_for_week(WEEK_PAGE_ID)

rows = []
for p in pages:
    props = p.get("properties", {})
    rows.append({
        "page_id": normalize_uuid(p.get("id","")),
        "name": _get_plain_text_title(props.get("Name")),
        "proposal_type": _get_select_name(props.get("Proposal Type")),
        "field": _get_select_name(props.get("Field")),
        "status": _get_select_name(props.get("Status")),
        "confidence": _get_number(props.get("Confidence")),
        "change_summary": _get_plain_text_rich(props.get("Change Summary")),
        "rationale": _get_plain_text_rich(props.get("Rationale")),
        "target_rel_count": len((props.get("Target") or {}).get("relation", [])),
    })

proposals_df = pd.DataFrame(rows)
print(f"✓ proposals_df built from Notion: {len(proposals_df)} rows")
if not proposals_df.empty:
    print(proposals_df[["name","proposal_type","field","status","confidence"]].head(10).to_string(index=False))

✓ proposals_df built from Notion: 46 rows
                                                                name  proposal_type           field   status  confidence
                         Masayoshi Son — 2026-02-08 — Keyword Update Keyword Update Search Keywords Proposed         0.8
                                   AMED — 2026-02-08 — Source Update  Source Update     Source URLs Proposed         0.8
OECD Science, Technology and Innovation — 2026-02-08 — Source Update  Source Update     Source URLs Proposed         0.8
                       Lightspeed India — 2026-02-08 — Source Update  Source Update     Source URLs Proposed         0.8
                        Eduardo Saverin — 2026-02-08 — Source Update  Source Update     Source URLs Proposed         0.8
                          Reid Hoffman — 2026-02-08 — Keyword Update Keyword Update Search Keywords Proposed         0.8
             a16z (Andreessen Horowitz) — 2026-02-08 — Source Update  Source Update     Source URLs Proposed   

In [46]:
# ============================================================
# Cell 14 — Summary visualization and export (CLEAN + ROBUST)
# ============================================================
# Overview:
#   Generate lightweight, robust summaries and exports for Weekly Targets Review:
#     - Single-figure visualizations (no subplots; simpler + clearer)
#     - CSV exports (evaluation summary + proposals)
#     - Text report export
#
# Inputs / Outputs:
#   Inputs:
#     - targets_df, events_df, metrics_df, evaluation_df (optional), proposals_df (optional)
#     - current_week (str), DAYS_LOOKBACK (int)
#     - MIN_SIGNAL_THRESHOLD, HIGH_NOISE_THRESHOLD (floats)
#   Outputs:
#     - PNG charts: effectiveness_dist, noise_dist, events_dist, targets_by_type, proposals_by_type (optional)
#     - CSV: target_effectiveness_summary_{current_week}.csv (if evaluation_df exists)
#     - CSV: proposals_{current_week}.csv (if proposals_df exists)
#     - TXT: weekly_target_review_report_{current_week}.txt
#
# Notes:
#   - No seaborn, no subplots (per your environment/style constraints).
#   - Defensive against missing dfs/columns.
#

import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime


print("Generating summary visualizations and final exports...\n")

# --- Resolve current_week safely ---
if "current_week" not in globals() or not current_week:
    # Option 1: derive from WEEK_PAGE (recommended if available)
    try:
        if "current_week_page" in globals() and current_week_page:
            # Example title: "Weekly Events Digest: 2026-02-02–2026-02-08"
            title = current_week_page.get("title", "")
            m = re.search(r"\d{4}-\d{2}-\d{2}.*\d{4}-\d{2}-\d{2}", title)
            current_week = m.group(0) if m else datetime.now().strftime("%Y-%m-%d")
        else:
            raise ValueError("no week page")
    except Exception:
        # Option 2: fallback to today
        current_week = datetime.now().strftime("%Y-%m-%d")

print(f"Using current_week = {current_week}")

def _safe_str(x) -> str:
    return "" if x is None else str(x)

def _ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

OUTPUT_DIR = "outputs/weekly_target_review"
_ensure_dir(OUTPUT_DIR)

def _savefig(fig, filename: str) -> str:
    path = os.path.join(OUTPUT_DIR, filename)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"✓ Saved: {path}")
    return path

def _has_cols(df: pd.DataFrame, cols: list[str]) -> bool:
    return df is not None and not df.empty and all(c in df.columns for c in cols)

# ----------------------------
# 1) Charts (single charts)
# ----------------------------

# (A) Effectiveness distribution
if "evaluation_df" in globals() and isinstance(evaluation_df, pd.DataFrame) and not evaluation_df.empty and "llm_effectiveness_score" in evaluation_df.columns:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    ax.hist(evaluation_df["llm_effectiveness_score"].dropna(), bins=20, edgecolor="black", alpha=0.7)
    ax.axvline(float(MIN_SIGNAL_THRESHOLD), linestyle="--", label=f"Min Threshold ({MIN_SIGNAL_THRESHOLD})")
    ax.set_title("Target Effectiveness Distribution (LLM)")
    ax.set_xlabel("LLM Effectiveness Score")
    ax.set_ylabel("Count")
    ax.grid(True, alpha=0.3)
    ax.legend()
    eff_png = _savefig(fig, f"effectiveness_dist_{current_week}.png")
else:
    eff_png = None
    print("ℹ Skipped effectiveness distribution (evaluation_df missing/empty or column missing)")

# (B) Noise distribution
if "evaluation_df" in globals() and isinstance(evaluation_df, pd.DataFrame) and not evaluation_df.empty and "llm_refined_noise_ratio" in evaluation_df.columns:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    ax.hist(evaluation_df["llm_refined_noise_ratio"].dropna(), bins=20, edgecolor="black", alpha=0.7)
    ax.axvline(float(HIGH_NOISE_THRESHOLD), linestyle="--", label=f"High Noise ({HIGH_NOISE_THRESHOLD})")
    ax.set_title("Target Noise Distribution (LLM refined)")
    ax.set_xlabel("Noise Ratio")
    ax.set_ylabel("Count")
    ax.grid(True, alpha=0.3)
    ax.legend()
    noise_png = _savefig(fig, f"noise_dist_{current_week}.png")
else:
    noise_png = None
    print("ℹ Skipped noise distribution (evaluation_df missing/empty or column missing)")

# (C) Event count distribution
if "metrics_df" in globals() and isinstance(metrics_df, pd.DataFrame) and not metrics_df.empty and "event_count" in metrics_df.columns:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    ax.hist(metrics_df["event_count"].dropna(), bins=20, edgecolor="black", alpha=0.7)
    ax.set_title(f"Events per Target Distribution ({DAYS_LOOKBACK} days)")
    ax.set_xlabel("Event Count")
    ax.set_ylabel("Count")
    ax.grid(True, alpha=0.3)
    events_png = _savefig(fig, f"events_per_target_{current_week}.png")
else:
    events_png = None
    print("ℹ Skipped events distribution (metrics_df missing/empty or column missing)")

# (D) Targets by type
if "targets_df" in globals() and isinstance(targets_df, pd.DataFrame) and not targets_df.empty and "type" in targets_df.columns:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    counts = targets_df["type"].fillna("Unknown").value_counts()
    ax.bar(counts.index.astype(str), counts.values, edgecolor="black")
    ax.set_title("Active Targets by Type")
    ax.set_xlabel("Target Type")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3, axis="y")
    type_png = _savefig(fig, f"targets_by_type_{current_week}.png")
else:
    type_png = None
    print("ℹ Skipped targets-by-type (targets_df missing/empty or column missing)")

# (E) Proposals by type (optional)
if "proposals_df" in globals() and isinstance(proposals_df, pd.DataFrame) and not proposals_df.empty and "proposal_type" in proposals_df.columns:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    pcounts = proposals_df["proposal_type"].fillna("Unknown").value_counts()
    ax.barh(pcounts.index.astype(str), pcounts.values, edgecolor="black")
    ax.set_title("Proposals by Type")
    ax.set_xlabel("Count")
    ax.set_ylabel("Proposal Type")
    ax.grid(True, alpha=0.3, axis="x")
    prop_png = _savefig(fig, f"proposals_by_type_{current_week}.png")
else:
    prop_png = None
    print("ℹ Skipped proposals-by-type (proposals_df missing/empty or column missing)")

# ----------------------------
# 2) CSV Exports
# ----------------------------

summary_filename = None
if "evaluation_df" in globals() and isinstance(evaluation_df, pd.DataFrame) and not evaluation_df.empty:
    # prefer these columns if present (support both naming styles)
    cols_map = [
        ("target_name", "Target Name"),
        ("target_type", "Type"),
        ("priority", "Priority"),
        ("cadence", "Cadence"),
        ("event_count", "Event Count"),
        ("computed_signal_quality", "Computed Signal Quality"),
        ("llm_effectiveness_score", "LLM Effectiveness"),
        ("llm_refined_noise_ratio", "LLM Noise Ratio"),
        ("llm_rationale", "LLM Rationale"),
    ]
    existing = [(c, out) for c, out in cols_map if c in evaluation_df.columns]
    if existing:
        summary_export = evaluation_df[[c for c, _ in existing]].copy()
        summary_export.columns = [out for _, out in existing]
        summary_filename = os.path.join(OUTPUT_DIR, f"target_effectiveness_summary_{current_week}.csv")
        summary_export.to_csv(summary_filename, index=False)
        print(f"✓ Summary exported: {summary_filename}")
    else:
        print("ℹ Skipped summary export: expected columns not found in evaluation_df")

export_filename = None
if "proposals_df" in globals() and isinstance(proposals_df, pd.DataFrame) and not proposals_df.empty:
    export_filename = os.path.join(OUTPUT_DIR, f"proposals_{current_week}.csv")
    proposals_df.to_csv(export_filename, index=False)
    print(f"✓ Proposals exported: {export_filename}")

# ----------------------------
# 3) Text Report Export
# ----------------------------

def _vc(df: pd.DataFrame, col: str) -> dict:
    if df is None or df.empty or col not in df.columns:
        return {}
    return df[col].fillna("Unknown").value_counts().to_dict()

def _metric(df: pd.DataFrame, col: str, fn) -> float | None:
    if df is None or df.empty or col not in df.columns:
        return None
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    if s.empty:
        return None
    return float(fn(s))

report_lines = []
report_lines.append("===== WEEKLY TARGET REVIEW SUMMARY =====")
report_lines.append(f"Week: {current_week}")
report_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append("")

# Active targets
report_lines.append("=== ACTIVE TARGETS ===")
report_lines.append(f"Total active targets: {len(targets_df) if 'targets_df' in globals() and isinstance(targets_df, pd.DataFrame) else 0}")
if "targets_df" in globals() and isinstance(targets_df, pd.DataFrame) and not targets_df.empty:
    report_lines.append(f"By Type: {_vc(targets_df, 'type')}")
    report_lines.append(f"By Priority: {_vc(targets_df, 'priority')}")
    report_lines.append(f"By Cadence: {_vc(targets_df, 'cadence')}")
report_lines.append("")

# Events
report_lines.append(f"=== EVENT ACTIVITY ({DAYS_LOOKBACK} days) ===")
report_lines.append(f"Total events: {len(events_df) if 'events_df' in globals() and isinstance(events_df, pd.DataFrame) else 0}")
if "events_df" in globals() and isinstance(events_df, pd.DataFrame) and not events_df.empty:
    if "target_count" in events_df.columns:
        report_lines.append(f"Events with target relations: {int((events_df['target_count'] > 0).sum())}")
    else:
        report_lines.append("Events with target relations: (target_count column missing)")
if "metrics_df" in globals() and isinstance(metrics_df, pd.DataFrame) and not metrics_df.empty and "event_count" in metrics_df.columns:
    report_lines.append(f"Targets with events: {int((metrics_df['event_count'] > 0).sum())}")
    report_lines.append(f"Targets with no events: {int((metrics_df['event_count'] == 0).sum())}")
report_lines.append("")

# Effectiveness
report_lines.append("=== EFFECTIVENESS METRICS ===")
mean_eff = _metric(evaluation_df if "evaluation_df" in globals() else None, "llm_effectiveness_score", np.mean)
med_eff = _metric(evaluation_df if "evaluation_df" in globals() else None, "llm_effectiveness_score", np.median)
mean_noise = _metric(evaluation_df if "evaluation_df" in globals() else None, "llm_refined_noise_ratio", np.mean)

if mean_eff is not None:
    report_lines.append(f"Mean effectiveness: {mean_eff:.3f}")
if med_eff is not None:
    report_lines.append(f"Median effectiveness: {med_eff:.3f}")
if mean_noise is not None:
    report_lines.append(f"Mean noise ratio: {mean_noise:.3f}")

if "evaluation_df" in globals() and isinstance(evaluation_df, pd.DataFrame) and not evaluation_df.empty:
    if "llm_effectiveness_score" in evaluation_df.columns:
        report_lines.append(f"Targets below signal threshold ({MIN_SIGNAL_THRESHOLD}): {int((evaluation_df['llm_effectiveness_score'] < MIN_SIGNAL_THRESHOLD).sum())}")
    if "llm_refined_noise_ratio" in evaluation_df.columns:
        report_lines.append(f"Targets above noise threshold ({HIGH_NOISE_THRESHOLD}): {int((evaluation_df['llm_refined_noise_ratio'] > HIGH_NOISE_THRESHOLD).sum())}")
report_lines.append("")

# Proposals
report_lines.append("=== PROPOSED ADJUSTMENTS ===")
pcount = len(proposals_df) if "proposals_df" in globals() and isinstance(proposals_df, pd.DataFrame) else 0
report_lines.append(f"Total proposals: {pcount}")
if "proposals_df" in globals() and isinstance(proposals_df, pd.DataFrame) and not proposals_df.empty:
    report_lines.append(f"By type: {_vc(proposals_df, 'proposal_type')}")
report_lines.append("")

def _pick_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None

if "evaluation_df" in globals() and isinstance(evaluation_df, pd.DataFrame) and not evaluation_df.empty and "llm_effectiveness_score" in evaluation_df.columns:
    name_col = _pick_col(evaluation_df, ["target_name", "Target Name", "name", "target"])
    type_col = _pick_col(evaluation_df, ["target_type", "Type", "type"])
    score_col = "llm_effectiveness_score"

    report_lines.append("=== TOP 5 PERFORMERS ===")
    top5 = evaluation_df.nlargest(5, score_col)
    for _, r in top5.iterrows():
        nm = _safe_str(r.get(name_col, "")) if name_col else ""
        tp = _safe_str(r.get(type_col, "")) if type_col else ""
        sc = float(r.get(score_col, 0.0))
        report_lines.append(f"- {nm} ({tp}): {sc:.3f}")

    report_lines.append("")
    report_lines.append("=== BOTTOM 5 PERFORMERS ===")
    bot5 = evaluation_df.nsmallest(5, score_col)
    for _, r in bot5.iterrows():
        nm = _safe_str(r.get(name_col, "")) if name_col else ""
        tp = _safe_str(r.get(type_col, "")) if type_col else ""
        sc = float(r.get(score_col, 0.0))
        report_lines.append(f"- {nm} ({tp}): {sc:.3f}")

    report_lines.append("")


# Next steps
report_lines.append("=== NEXT STEPS ===")
report_lines.append("1. Review proposals in Weekly Target Update DB")
report_lines.append("2. Approve/reject proposals based on strategic priorities")
report_lines.append("3. Apply approved changes to Monitoring Targets DB")
report_lines.append("4. Monitor impact over next review cycle")
report_lines.append("")
report_lines.append("========================================")

report_text = "\n".join(report_lines)

print(report_text)

report_filename = os.path.join(OUTPUT_DIR, f"weekly_target_review_report_{current_week}.txt")
with open(report_filename, "w", encoding="utf-8") as f:
    f.write(report_text)
print(f"\n✓ Text report saved: {report_filename}")

print(f"\n{'='*60}")
print("WEEKLY TARGET REVIEW COMPLETE")
print(f"{'='*60}")
print(f"Week: {current_week}")
print(f"Active Targets: {len(targets_df) if 'targets_df' in globals() and isinstance(targets_df, pd.DataFrame) else 0}")
print(f"Events Analyzed: {len(events_df) if 'events_df' in globals() and isinstance(events_df, pd.DataFrame) else 0}")
print(f"Proposals Generated: {len(proposals_df) if 'proposals_df' in globals() and isinstance(proposals_df, pd.DataFrame) else 0}")
print("\nExports:")
print(f"  - Proposals CSV: {export_filename if export_filename else 'N/A'}")
print(f"  - Summary CSV: {summary_filename if summary_filename else 'N/A'}")
print(f"  - Charts dir: {OUTPUT_DIR}")
print(f"  - Text Report: {report_filename}")
print(f"{'='*60}")


Generating summary visualizations and final exports...

Using current_week = 2026-02-08
✓ Saved: outputs/weekly_target_review/effectiveness_dist_2026-02-08.png
✓ Saved: outputs/weekly_target_review/noise_dist_2026-02-08.png
ℹ Skipped events distribution (metrics_df missing/empty or column missing)
✓ Saved: outputs/weekly_target_review/targets_by_type_2026-02-08.png
✓ Saved: outputs/weekly_target_review/proposals_by_type_2026-02-08.png
✓ Summary exported: outputs/weekly_target_review/target_effectiveness_summary_2026-02-08.csv
✓ Proposals exported: outputs/weekly_target_review/proposals_2026-02-08.csv
===== WEEKLY TARGET REVIEW SUMMARY =====
Week: 2026-02-08
Generated: 2026-02-08 11:58:38

=== ACTIVE TARGETS ===
Total active targets: 32
By Type: {'POLICY': 16, 'VC': 9, 'PEOPLE': 6, 'STARTUP': 1}
By Priority: {'High': 23, 'Medium': 9}
By Cadence: {'DAILY': 17, 'WEEKLY': 15}

=== EVENT ACTIVITY (7 days) ===
Total events: 63
Events with target relations: 63

=== EFFECTIVENESS METRICS ===
M